# Embeddings Cache → Outlier Scores Pipeline

This notebook implements a **step-by-step** workflow that starts from long-format parquet extractions and produces per-sample anomaly/outlier scores using Presto embeddings.

**Full pipeline:**
```
Long-format parquets (.parquet or .geoparquet)
  → Wide time-series parquets (one per input file, process_parquet)
  → Merged wide parquet (single Arrow-streamed file)
  → DuckDB embeddings cache (Presto embeddings, keyed by sample_id + model_hash)
  → Outlier scores — LANDCOVER10 and CROPTYPE24 (run_pipeline from worldcereal.train.anomaly)
  → Merged scores DataFrame (one row per sample_id)
  → Scores appended back to input parquets (in-place for geoparquet, or to OUTPUT_LONG_DIR)
  → Scores appended back to merged wide parquet (Arrow-streamed, memory-safe)
```

**Two input layouts are supported — controlled by `INPUT_FORMAT` in section 1:**

| `INPUT_FORMAT` | Layout | Write-back |
|---|---|---|
| `"geoparquet"` | Flat folder of per-dataset `.geoparquet` files (VM) | **In-place**, geo metadata preserved |
| `"parquet"` | Nested hive-partitioned `.parquet` files (HPC) | Separate `OUTPUT_LONG_DIR` |

> **Note:** DuckDB may throw a lock error if the same `.duckdb` file is open for writing in another process or kernel.  Use a different DB path or close other connections before running the embeddings cells.


## ⚠️ Updated detector — read before running

The scoring logic changed. This notebook has been updated to match; the notes
below explain what is different and which knobs you may want to change.

**A flag now requires two conditions, not one.**
Every within-slice score is percentile-normalised *per slice*, so on its own it
cannot tell *the most unusual point of a clean slice* from *a mislabelled point*
— the top ~2 % of every slice cleared the old escalation thresholds whether or
not the slice held a single real error. And the old `median + k·MAD` gate was a
knife-edge: on synthetic data it flagged **0 %** at both 2 % and 30 % true
contamination, but 9 % at 10 %, because a contaminated slice inflates its own
median and MAD. So a sample must now be:

1. **locally unusual** — `threshold_mode="stable_mad"` keeps the slice median as
   the local reference but takes the *dispersion* from a cross-slice null, which
   a single slice cannot inflate; and
2. **absolutely unusual** — `abs_z >= abs_z_k` robust sigma against a null pooled
   across many slices of the same class, built from one summary statistic per
   slice so a dirty slice cannot calibrate its own errors away.

Set `require_absolute=False` to reproduce the old relative-only behaviour.

**`group_cols` now defaults to `["ref_id"]`.**
Slices are scored per source dataset, so one dataset's labelling convention
cannot contaminate another's reference cloud. **Pass `group_cols=[]` to pool
across datasets** — the previous behaviour of these notebooks. Pooling lets a
dataset digitised against a different legend either mask itself (if it is large)
or get flagged wholesale (if it is small). The *context* used for the kNN-purity
and alt-class-margin signals stays geographic either way; override it with
`context_group_cols` only if you know you want per-dataset context.

**New flag values.** `*_anomaly_flag` can now also be:

| value | meaning |
|---|---|
| `unscored` | slice too small to score — **not** the same as `normal` |
| `unscorable` | embedding failed the quality gate (zero-norm, non-finite, duplicate id); see `quality_reason` |
| `unmapped` | `ewoc_code` absent from the legend — a coverage gap |
| `skipped` | held out by `skip_classes` |

These are terminal states recording that the detector *could not form an
opinion*. Any code that ranks flags must not fold them into `normal`; the helper
cells below use `_NON_JUDGED_FLAGS` for this. They are also never dropped by the
`experiments/scenarios.py` drop sets — "we did not look" must not remove a
training sample.

**Escalation is stricter.** `suspect` / `candidate` need agreement across signals
that measure different things (absolute distance, kNN label purity, alt-class
margin, within-slice rank). A point whose neighbours overwhelmingly share its
label is capped at `flagged` (`purity_veto`), and where a context contains a
single label — no alternative class to have been confused with — escalation is
capped at `flagged` too.

**Evidence is kept.** The review parquets now carry `abs_z`, `cosine_distance`,
`knn_distance`, `neighbourhood_offset`, `knn_same_label_frac_ctx`,
`alt_margin_ctx`, `escalation_votes`, `weak_support`, `purity_veto`,
`corroborated` and `quality_reason`, so a flag can be audited against a basemap
instead of taken on trust.

**Update — heavy slice contamination.** Defaults changed again after measuring
the 30–45 % regime end-to-end. The blocker was the *scale*, not the centroid:
the cross-slice null took the median of per-slice **MADs**, and a MAD is
widened by the very right-side errors it measures against — so when every slice
of a class carried 30–45 % errors the null inflated with them and nothing
cleared the gate. `null_scale_estimator="left_tail"` measures the spread as
`median − q25`, from the clean left half only.

| slice contamination | recall before | recall now |
|---|---|---|
| 20 % | 0.992 | 0.994 |
| 30 % | 0.798 | 0.985 |
| 40 % | 0.140 | 0.902 |
| 45 % | 0.084 | 0.256 |

…at a clean-data false-positive rate of 1.11 %, slightly **below** the previous
1.17 %. Precision stays above 0.98 throughout. New defaults: `mad_k=3.3`,
`abs_z_k=3.3`, `centroid_trim=0.45` (the trim must be ≥ the worst contamination
you expect), `null_scale_estimator="left_tail"`. Pass
`null_scale_estimator="mad"` for the legacy ablation.

**Update — slicing defaults.** `group_cols` is back to `[]` (pool across source
datasets). Comparing a sample against the same class in the same locality
*whoever digitised it* is the assumption the method rests on; splitting by
`ref_id` discards it, fragments small datasets below `min_slice_size` into
`unscored`, and makes a wholly-mislabelled dataset undetectable (it becomes its
own self-consistent slice: measured `abs_z` −0.15 and 0/360 flagged, versus
`abs_z` 5.45 and 339/360 when pooled).

The cross-slice null is now conditioned on `h3_effective_level`. In adaptive
mode a sparse-region slice can be an L2 cell (~332 km across) while a dense one
is L4 (~47 km) — the coarser cell spans several agro-ecological zones and so
disperses more *legitimately*. Measured on clean data holding both: pooling gave
coarse slices a **10.4× higher false-positive rate** (5.44 % vs 0.53 %);
conditioning equalised them (0.50 % vs 0.66 %). It also removes the recall that
bias was manufacturing there (coarse-slice recall 0.64 → 0.36 at 20 %
contamination, precision 0.95 → 0.99). Pass `null_extra_keys=[]` to pool.

**Correction — the null is now localised, not global.** The previous build
pooled the cross-slice null per class *globally* (conditioned only on H3
resolution). That was wrong for a global collection: the null asks "how far
from its own slice centroid does a typical sample of this class sit?", and the
honest answer differs region to region — wheat in a uniform monoculture
disperses far less around its local centroid than wheat in a fragmented
smallholder landscape. A globally pooled null is set by whichever landscape
contributes the most slices, so every more-variable region looked anomalous as
a whole. H3 resolution could not capture this: those regions often resolve at
the same level.

The null is now conditioned on a spatial region key (`h3_null_region`, the
slice cell's H3 L1 parent), shrunk toward the global null by
`w = n_slices / (n_slices + null_shrink_k)` so thin regions degrade smoothly.

Measured on clean data at 3, 5, 12 and 25 slices per region, the regional null
gave roughly **a third the false-positive rate** of a pooled one (0.7–0.8 % vs
2.0–2.4 %), and cut the spread *across* regions from 3.85 % to 0.78 %.

It is a trade, not a free win: at 10 % contamination precision rises
0.90 → 0.97 while recall falls 0.89 → 0.79, because much of what the pooled
null was "finding" was the regional bias rather than real errors. Set
`null_extra_keys=[]` and `null_region_level=None` to pool globally again.

**Update — the null is a ladder, and it now knows the slice's H3 resolution.**
Landcover improved after the previous change but croptype regressed, and the
cause was code-level. The null region was a **fixed** L1 parent regardless of
the slice's own resolution, so a `h3_level=[2, 3, 4]` croptype run pooled L2
(86,802 km²), L3 (12,393 km²) and L4 (1,770 km²) slices into one null group,
while a `[2, 3]` landcover run — almost all L3 in dense regions — barely felt
it. A bigger cell spans more legitimate variation, so the tight L4 slices
inherited a scale set partly by the coarse L2 ones and their real errors fell
under the gate.

Two changes:

1. The region is now **relative** to the slice's own resolution,
   `max(slice_res - null_region_offset, null_region_min_level)` (offset 2, floor
   L1), so every slice is calibrated against roughly the same *number* of
   sibling cells rather than the same absolute area.
2. The slice's own resolution is a null key in its own right (`h3_null_res`),
   because the region key alone does not separate resolutions — floored at L1,
   one region still holds both L2 and L3 slices.

`null_extra_keys` is now read as a **nesting**, coarsest first. A null is
estimated at every prefix — `(class)`, `(class, region)`,
`(class, region, resolution)` — and each row is calibrated against the finest
group that exists for it, each depth shrunk toward its own parent rather than
the flat global. Without that ladder the extra key would strand exactly the
groups it thins: a rare crop in a fine cell would fall all the way back to a
resolution-blind global null.

Measured end to end on a co-located L2/L3/L4 croptype run with 15 % planted
errors:

| null keys | recall | clean FP |
|---|---|---|
| class only | 30.9 % | 0.368 % |
| class + fixed L1 region (the regressed build) | 30.9 % | 0.368 % |
| class + relative region | 37.4 % | 0.150 % |
| **class + region + resolution** | **38.5 %** | **0.109 %** |

The two keys fix different levels. The **relative region** rescues L4 (recall
32 % → 71 %): an L4 slice's region is an L2 cell that holds only other L4
slices, so the region key separates resolutions there by itself. The
**resolution key** does the rest — L2 and L3 slices share one L1 region, and
separating them lifts L3 recall 25 % → 31 % while halving L2's false positives
0.37 % → 0.20 %.

L2 recall falls throughout (23 % → 13 %) as part of the same trade: the old
scheme was manufacturing detections there from an under-estimated scale and
paying 0.90 % false positives for them.

For **LANDCOVER10** (`h3_level=[2, 3]`) both levels floor to the same L1
region, so the region key alone changes nothing and only the resolution key
bites: recall flat (24.8 % → 25.0 %) at half the false positives (0.252 % →
0.112 %), with L3 recall 28 % → 33 %. The same defaults serve both domains — no
landcover-specific setting is needed.

The run prints a depth histogram; a large share of rows at depth 0 or 1 means
the keys are too fine for this collection's density. A key that turns out not
to subdivide anything at all (a fixed single-resolution run, where
`h3_null_res` is constant) is dropped automatically, with a printed note.

**Expect different counts from previous runs.** Re-run end-to-end rather than
mixing old and new outputs in the same merge.


## Notebook structure

| # | Section | Description |
|---|---------|-------------|
| 1 | **Parameters** | All user-configurable paths, knobs, and **`PIPELINE_MODE`** (`"rerun"` or `"update"`) |
| 2 | **Discover inputs** | List long-format parquet files; count unique `(ref_id, sample_id)` pairs |
| 3 | **Long → Wide conversion** | Run `process_parquet` on each long file; write per-file wide parquets |
| 4 | **Merge wide parquets** | Stream-merge all wide parquets into a single file |
| 5 | **Populate embeddings cache** | Compute Presto embeddings and write to DuckDB (skips already-cached samples) |
| 6 | **Load class mappings** | Fetch LANDCOVER / CROPTYPE label legend from SharePoint |
| 7 | **Scoring — rerun mode** | Full scoring: run LC10 and CTY24 pipelines over the entire DuckDB cache |
| 7u | **Scoring — update mode** | Incremental: per-domain unscored detection → H3 impact zone → load only affected embeddings → score |
| 8 | **Merge LC10 + CTY24 scores** | Outer-join both score tables on `(ref_id, sample_id)` |
| 9 | **Write scores back — long parquets** | Broadcast anomaly columns to every row in the long-format files |
| 10 | **Write scores back — merged wide parquet** | Arrow-streaming join/update of scores into the merged wide parquet |
| — | **Optional: run pipeline as a script** | Single-command equivalent (for SLURM / screen) |
| 11 | **Explore embeddings cache** | Quick stats, random sample, PCA scatter plot |
| 12 | **DB comparison utilities** | Diagnostic cells for comparing row counts across DuckDB versions |

> **Which mode to use?**
> - **`rerun`**: Initial run, or when you want to re-score everything from scratch (HPC with plenty of RAM)
> - **`update`**: After adding new labelled datasets to an existing scored collection (VM-friendly, memory-efficient)


In [ ]:
# ============================================================
# SMOKE TEST OVERRIDE  — set USE_SMOKE_TEST = True to run the
# full pipeline on 5 tiny files instead of the full dataset.
# Set to False to use the production paths in the next cell.
# ============================================================
USE_SMOKE_TEST = False

if USE_SMOKE_TEST:
    from pathlib import Path
    _SMOKE = Path("/path/to/TestFolder/wc_outliers/data_for_outlier")

    # For smoke test we use fresh paths so we don't clobber the rerun results
    _SMOKE_NB = _SMOKE / "notebook_run"
    _SMOKE_NB.mkdir(parents=True, exist_ok=True)

    SMOKE_OVERRIDES = dict(
        suffix             = "_nb_smoke",
        PIPELINE_MODE      = "rerun",       # change to "update" to test that path
        INPUT_FORMAT       = "geoparquet",
        INPUT_LONG_DIR     = _SMOKE / "input",
        WIDE_DIR           = _SMOKE_NB / "wide",
        MERGED_WIDE_PATH   = _SMOKE_NB / "merged_wide.parquet",
        EMBEDDINGS_DB_PATH = _SMOKE_NB / "embeddings.duckdb",
        OUTPUT_LONG_DIR    = _SMOKE_NB / "output_long",
        PARQUET_GLOB       = "*.geoparquet",
        MAX_LONG_FILES     = None,
        OVERWRITE_WIDE     = False,
        OVERWRITE_MERGED   = False,
        BATCH_SIZE         = 512,
        NUM_WORKERS        = 0,
        PREMATCH           = False,
    )

    # Override class mappings path (no SharePoint needed)
    import os
    os.environ["SMOKE_TEST_CLASS_MAPPINGS"] = str(
        Path(os.__file__).parent.parent /
        "lib/python3.11/site-packages/worldcereal/data/croptype_mappings/class_mappings.json"
    )
    # Find the actual path
    import glob as _glob
    _cm_candidates = _glob.glob(
        "/path/to/.conda/envs/wc_outliers/lib/python*/site-packages/"
        "worldcereal/data/croptype_mappings/class_mappings.json"
    )
    SMOKE_CLASS_MAPPINGS_JSON = Path(_cm_candidates[0]) if _cm_candidates else None
    print(f"Smoke test class mappings: {SMOKE_CLASS_MAPPINGS_JSON}")
    print(f"Smoke test input dir     : {SMOKE_OVERRIDES['INPUT_LONG_DIR']}")
    print(f"Smoke test output dir    : {SMOKE_OVERRIDES['OUTPUT_LONG_DIR']}")
    print("USE_SMOKE_TEST=True  production paths in the next cell will be overridden.")
else:
    SMOKE_OVERRIDES = {}
    SMOKE_CLASS_MAPPINGS_JSON = None
    print("USE_SMOKE_TEST=False  using production paths from the next cell.")

## 1) Parameters

Edit the variables below before running any other cells.  Everything else in the notebook reads from these names.

In [ ]:

from __future__ import annotations

import gc
import os
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import torch
from loguru import logger
from prometheo.models import Presto
from tqdm.auto import tqdm

from EBA_detector.anomaly import run_pipeline
from EBA_detector.anomaly_utils import (
    ANOMALY_COLUMNS as _ALL_ANOMALY_COLUMNS,
    LC10_ANOMALY_COLUMNS,
    CTY24_ANOMALY_COLUMNS,
    find_unscored_samples,
    compute_impact_zone,
    load_affected_embeddings_from_cache,
    merge_scores_to_long_parquets,
)
from EBA_detector.embeddings_cache import compute_embeddings, get_model_hash
from worldcereal.utils.timeseries import process_parquet

# ============================================================
# IDENTITY
# ============================================================
suffix = "_new_model"

# ============================================================
# PIPELINE MODE
# ============================================================
PIPELINE_MODE = "rerun"   # ← "rerun" or "update"

# ============================================================
# INPUT FORMAT
# ============================================================
INPUT_FORMAT = "geoparquet"   # ← "parquet" or "geoparquet"

# ============================================================
# PATHS  —  VM production paths
# ============================================================
_VM_DATA = Path("/path/to/TestFolder/wc_outliers/data_for_outlier")

INPUT_LONG_DIR = _VM_DATA / "MERGED_PARQUETS_PHASEII_WITH_ANOMALY"

WIDE_DIR         = _VM_DATA / f"CACHED_WIDE_MERGED/cached_wide_merged/cached_wide_parquets{suffix}"
WIDE_SUFFIX      = "_ppq"
OVERWRITE_WIDE   = False
OVERWRITE_MERGED = False

MERGED_WIDE_PATH = (
    _VM_DATA / f"CACHED_WIDE_MERGED/cached_wide_merged/worldcereal_all_extractions_wide_month{suffix}.parquet"
)

EMBEDDINGS_DB_PATH = (
    _VM_DATA / f"EMBEDDINGS_CACHE/embeddings_cache_LANDCOVER10_updated{suffix}.duckdb"
)

PRESTO_URL = (
    "/projects/worldcereal/models/cbutsko/WorldCerealPresto-GLOBAL-LC10CT24-RegionalMultipliersLCCT-MaskCap-month-augment=True-balance=TruePerBin-SpatialBin5.0deg-timeexplicit=True-masking=enabled-ema0.2-clamp=0.2-8.0-run=202606121328/WorldCerealPresto-GLOBAL-LC10CT24-RegionalMultipliersLCCT-MaskCap-month-augment=True-balance=TruePerBin-SpatialBin5.0deg-timeexplicit=True-masking=enabled-ema0.2-clamp=0.2-8.0-run=202606121328_encoder.pt"
)

_OUTLIERS_DIR = _VM_DATA / "EMBEDDINGS_CACHE"

OUT_LC10_DIR  = _OUTLIERS_DIR / f"h3levels23_10kmax_mad4_jsonLC10{suffix}"
OUT_CTY24_DIR = _OUTLIERS_DIR / f"h3levels234_5kmax_mad4_jsonCTY24{suffix}"

LC10_SAMPLES_PATH = str(OUT_LC10_DIR / f"h3levels23_10kmax_mad4_jsonLC10{suffix}.parquet")
LC10_SUMMARY_PATH = str(OUT_LC10_DIR / f"h3levels23_10kmax_mad4_jsonLC10_summary{suffix}.parquet")

CTY24_SAMPLES_PATH = str(OUT_CTY24_DIR / f"h3levels234_5kmax_mad4_jsonCTY24{suffix}.parquet")
CTY24_SUMMARY_PATH = str(OUT_CTY24_DIR / f"h3levels234_5kmax_mad4_jsonCTY24_summary{suffix}.parquet")

ANOMALY_COLS = [
    "CTY24_confidence_nonoutlier",
    "CTY24_anomaly_flag",
    "outlier_CTY24_cls",
    "LC10_confidence_nonoutlier",
    "LC10_anomaly_flag",
    "outlier_LC10_cls",
]

MERGED_SCORES_PATH = _OUTLIERS_DIR / f"merged_LC10_CTY24_flagged_gdf{suffix}.parquet"

if INPUT_FORMAT == "geoparquet":
    OUTPUT_LONG_DIR = INPUT_LONG_DIR
    PARQUET_GLOB    = "*.geoparquet"
else:
    _HPC_HOME = Path("/path/to")
    INPUT_LONG_DIR  = _HPC_HOME / "projects/worldcereal/data/worldcereal_all_extractions.parquet"
    OUTPUT_LONG_DIR = _HPC_HOME / f"projects/worldcereal/data/worldcereal_all_extractions_with_anomalies{suffix}.parquet"
    PARQUET_GLOB    = "**/*.parquet"

OUTPUT_WIDE_WITH_SCORES_PATH = MERGED_WIDE_PATH.parent / (MERGED_WIDE_PATH.stem + "_with_anomalies.parquet")

# ============================================================
# PIPELINE KNOBS
# ============================================================
FREQ = "month"
REQUIRED_MIN_TIMESTEPS = None
USE_VALID_TIME = True
MIN_EDGE_BUFFER = 1
MAX_TIMESTEPS_TRIM = 18
WIDE_ENGINE = "pyarrow"
WIDE_COMPRESSION = "snappy"
MAX_LONG_FILES = None

MERGE_BATCH_ROWS    = 100_000
MERGE_ROW_GROUP_SIZE = 100_000
MERGE_COMPRESSION   = "zstd"

BATCH_SIZE        = 4096*2
NUM_WORKERS       = 8
PARQUET_BATCH_ROWS = 300_000
FORCE_RECOMPUTE   = False
PREMATCH          = True

NEIGHBOUR_RINGS = 1

# ============================================================
# APPLY SMOKE TEST OVERRIDES (if USE_SMOKE_TEST=True above)
# ============================================================
if "SMOKE_OVERRIDES" in dir() and SMOKE_OVERRIDES:
    _ov = SMOKE_OVERRIDES
    suffix             = _ov.get("suffix", suffix)
    PIPELINE_MODE      = _ov.get("PIPELINE_MODE", PIPELINE_MODE)
    INPUT_FORMAT       = _ov.get("INPUT_FORMAT", INPUT_FORMAT)
    INPUT_LONG_DIR     = _ov.get("INPUT_LONG_DIR", INPUT_LONG_DIR)
    WIDE_DIR           = _ov.get("WIDE_DIR", WIDE_DIR)
    MERGED_WIDE_PATH   = _ov.get("MERGED_WIDE_PATH", MERGED_WIDE_PATH)
    EMBEDDINGS_DB_PATH = _ov.get("EMBEDDINGS_DB_PATH", EMBEDDINGS_DB_PATH)
    OUTPUT_LONG_DIR    = _ov.get("OUTPUT_LONG_DIR", OUTPUT_LONG_DIR)
    PARQUET_GLOB       = _ov.get("PARQUET_GLOB", PARQUET_GLOB)
    MAX_LONG_FILES     = _ov.get("MAX_LONG_FILES", MAX_LONG_FILES)
    OVERWRITE_WIDE     = _ov.get("OVERWRITE_WIDE", OVERWRITE_WIDE)
    OVERWRITE_MERGED   = _ov.get("OVERWRITE_MERGED", OVERWRITE_MERGED)
    BATCH_SIZE         = _ov.get("BATCH_SIZE", BATCH_SIZE)
    NUM_WORKERS        = _ov.get("NUM_WORKERS", NUM_WORKERS)
    PREMATCH           = _ov.get("PREMATCH", PREMATCH)
    # Smoke test output dirs under WIDE_DIR parent
    _smoke_base = WIDE_DIR.parent
    OUT_LC10_DIR  = _smoke_base / f"lc10{suffix}"
    OUT_CTY24_DIR = _smoke_base / f"cty24{suffix}"
    LC10_SAMPLES_PATH = str(OUT_LC10_DIR / f"lc10{suffix}.parquet")
    LC10_SUMMARY_PATH = str(OUT_LC10_DIR / f"lc10_summary{suffix}.parquet")
    CTY24_SAMPLES_PATH = str(OUT_CTY24_DIR / f"cty24{suffix}.parquet")
    CTY24_SUMMARY_PATH = str(OUT_CTY24_DIR / f"cty24_summary{suffix}.parquet")
    MERGED_SCORES_PATH = _smoke_base / f"merged_LC10_CTY24{suffix}.parquet"
    OUTPUT_WIDE_WITH_SCORES_PATH = MERGED_WIDE_PATH.parent / (MERGED_WIDE_PATH.stem + "_with_anomalies.parquet")
    print("[SMOKE TEST] Overrides applied.")

# ============================================================
# SANITY CHECKS
# ============================================================
OUTPUT_LONG_DIR.mkdir(parents=True, exist_ok=True)
OUT_LC10_DIR.mkdir(parents=True, exist_ok=True)
OUT_CTY24_DIR.mkdir(parents=True, exist_ok=True)

assert INPUT_LONG_DIR.exists(), f"Input path not found: {INPUT_LONG_DIR}"

print(f"Pipeline mode  : {PIPELINE_MODE}")
print(f"Input format   : {INPUT_FORMAT}")
print(f"Input long dir : {INPUT_LONG_DIR}")
print(f"Parquet glob   : {PARQUET_GLOB}")
print(f"Wide dir       : {WIDE_DIR}")
print(f"Merged wide    : {MERGED_WIDE_PATH}")
print(f"Embeddings DB  : {EMBEDDINGS_DB_PATH}")
print(f"Output long dir: {OUTPUT_LONG_DIR}  {'(in-place)' if OUTPUT_LONG_DIR == INPUT_LONG_DIR else ''}")
if PIPELINE_MODE == "update":
    print(f"Neighbour rings: {NEIGHBOUR_RINGS}")


## 2) Discover long-format parquet inputs

List all long-format parquet files matched by `PARQUET_GLOB` and count the unique `(ref_id, sample_id)` pairs across all files.  This is a read-only step — it does not produce any output files.

In [ ]:
raw_files = sorted(INPUT_LONG_DIR.glob(PARQUET_GLOB))
if MAX_LONG_FILES:
    raw_files = raw_files[:MAX_LONG_FILES]
print(f"Found {len(raw_files)} long-format parquet files (glob: '{PARQUET_GLOB}').")

wide_files = sorted(WIDE_DIR.glob("*.parquet"))
print(f"Found {len(wide_files)} wide-format parquet files in {WIDE_DIR}")
# all_ref_ids_samples: set = set()
# for f in tqdm(raw_files, desc="Scanning long parquets", unit="file"):
#     df = pd.read_parquet(f, columns=["ref_id", "sample_id"])
#     ref_id_sample_pairs = set(zip(df["ref_id"], df["sample_id"]))
#     all_ref_ids_samples.update(ref_id_sample_pairs)

# print(f"Found {len(all_ref_ids_samples):,} unique (ref_id, sample_id) pairs.")

# check how many of the geoparquet files have already been scored (i.e. contain the anomaly columns)
total_scored = 0
for f in tqdm(raw_files, desc="Checking scored files", unit="file"):
    try:
        table = pq.read_table(f, columns=ANOMALY_COLS)
        if all(col in table.column_names for col in ANOMALY_COLS):
            # logger.info(f"File {f} already contains anomaly columns.")
            # also check if the anomaly columns contain all nulls (i.e. not scored yet)
            anomaly_cols = [table[col] for col in ANOMALY_COLS]
            if all(pc.all(pc.is_null(col)).as_py() for col in anomaly_cols):
                logger.info(f"File {f} contains anomaly columns but they are all null (not scored yet).")
            total_scored += 1
        else:
            logger.info(f"File {f} is missing some anomaly columns.")
    except Exception as e:
        logger.warning(f"Could not read anomaly columns from {f}: {e}")

logger.info(f"{len(raw_files)-total_scored} files do not contain anomaly columns or are not scored yet.")

## 3) Convert long → wide via `process_parquet`

For every long-format parquet file, call `worldcereal.utils.timeseries.process_parquet` to pivot the stacked time-series rows into a wide format (one row per sample, one column per band×timestep).  Output files land in `WIDE_DIR` with the `WIDE_SUFFIX` suffix.

Already-converted files are skipped unless `OVERWRITE_WIDE = True`.

In [ ]:
def wide_out_path(wide_dir: Path, raw_path: Path, suffix: str = "_ppq") -> Path:
    # Strip ALL extensions so both .parquet and .geoparquet are handled uniformly
    # e.g. 2019_BGR_Eurocrops_POLY_110.geoparquet → 2019_BGR_Eurocrops_POLY_110_ppq.parquet
    stem = raw_path.name
    for ext in (".geoparquet", ".parquet"):
        if stem.endswith(ext):
            stem = stem[: -len(ext)]
            break
    return wide_dir / f"{stem}{suffix}.parquet"

WIDE_DIR.mkdir(parents=True, exist_ok=True)
already_processed = 0
wide_files: list[Path] = []
empty_long: list[Path] = []
errored_long: list[Path] = []

for pf in tqdm(raw_files, desc="process_parquet", unit="file"):
    out_path = wide_out_path(WIDE_DIR, pf, suffix=WIDE_SUFFIX)
    if out_path.exists() and not OVERWRITE_WIDE:
        already_processed += 1
        wide_files.append(out_path)
        continue
    try:
        df_long = pd.read_parquet(pf)
        # Some upstream outputs store sample_id in the index instead of a column
        if "sample_id" not in df_long.columns and "sample_id" in df_long.index.names:
            print("Resetting index to get sample_id column")
            df_long.reset_index(drop=True, inplace=True)
        df_wide = process_parquet(
            df_long,
            freq=FREQ,
            required_min_timesteps=REQUIRED_MIN_TIMESTEPS,
            use_valid_time=USE_VALID_TIME,
            min_edge_buffer=MIN_EDGE_BUFFER,
            max_timesteps_trim=MAX_TIMESTEPS_TRIM,
        )
    except Exception as e:
        print(f"ERROR processing {pf}: {type(e).__name__}: {e}")
        errored_long.append(pf)
        raise e

    if df_wide.empty:
        empty_long.append(pf)
        continue

    df_wide = df_wide.reset_index()
    df_wide.sort_values("sample_id", inplace=True)
    df_wide.to_parquet(
        out_path,
        engine=WIDE_ENGINE,
        compression=WIDE_COMPRESSION,
        index=False,
    )
    wide_files.append(out_path)
    del df_long, df_wide
    gc.collect()

print(f"Already processed (skipped) : {already_processed}")
print(f"Produced / available        : {len(wide_files)}")
if empty_long:
    print(f"Empty after process_parquet : {len(empty_long)}")
if errored_long:
    print(f"Errors during conversion    : {len(errored_long)}")
display([str(p) for p in wide_files[:10]])


In [ ]:
# -----------------------------------------------------------------------
# OPTIONAL: Backfill missing wide files from an earlier run
#
# If a previous processing run (e.g. with different SCL settings) already
# produced wide parquets for most files and only a few are missing in the
# new directory, this cell fills the gap by borrowing the earlier files
# instead of reprocessing from scratch.
#
# Uncomment and adjust the directory paths as needed.
# -----------------------------------------------------------------------

# earlier_dir = Path("/projects/worldcereal/data/cached_wide_merged/cached_wide_parquets")
# new_dir     = Path("/projects/worldcereal/data/cached_wide_merged/cached_wide_parquets_raw_scl")

# wide_files_earlier_all = list(earlier_dir.glob("*.parquet"))
# wide_files_new_all     = list(new_dir.glob("*.parquet"))

# # Compare by filename stem only (ignore directory)
# wide_files_earlier_stems = {p.stem: p for p in wide_files_earlier_all}
# wide_files_new_stems     = {p.stem: p for p in wide_files_new_all}

# missing_stems = set(wide_files_earlier_stems.keys()) - set(wide_files_new_stems.keys())
# print(f"Missing wide files from earlier runs : {len(missing_stems)}")
# print(f"New wide files (current run)         : {len(wide_files_new_all)}")

# # Resolve paths for the missing files (taken from the earlier folder)
# missing_wide_files = [wide_files_earlier_stems[stem] for stem in sorted(missing_stems)]

# # Override wide_files: new run files + backfilled missing
# wide_files = sorted(wide_files_new_all) + missing_wide_files
# print(f"Total wide files (new + backfilled)  : {len(wide_files)}")

## 4) Merge wide parquets into one (Arrow streaming)

Streams all per-file wide parquets into a single merged parquet file using PyArrow.  The implementation handles minor schema differences across files (different columns or types) by computing a unified target schema and casting/null-filling as needed.

This step is **memory-safe**: files are read in row-group batches and written incrementally; the full dataset is never loaded at once.

Output goes to `MERGED_WIDE_PATH`.  Skipped if the file already exists and `OVERWRITE_MERGED = False`.

In [ ]:
def _is_numeric(t: pa.DataType) -> bool:
    return pa.types.is_integer(t) or pa.types.is_floating(t) or pa.types.is_decimal(t)


def _build_target_schema(files: list[Path]) -> pa.Schema:
    """Build a unified Arrow schema from the union of all file schemas.

    When the same column has multiple types across files, the resolution
    rules are:
    - Any numeric type  → float32
    - Any timestamp     → timestamp("us")
    - Any bool present  → bool_
    - Anything else     → string
    """
    type_map: dict[str, set[pa.DataType]] = {}
    for f in files:
        sch = pq.ParquetFile(str(f)).schema_arrow
        for field in sch:
            type_map.setdefault(field.name, set()).add(field.type)

    fields: list[pa.Field] = []
    for name, typeset in sorted(type_map.items()):
        if len(typeset) == 1:
            fields.append(pa.field(name, next(iter(typeset))))
            continue
        if any(_is_numeric(t) for t in typeset):
            fields.append(pa.field(name, pa.float32()))
        elif any(pa.types.is_timestamp(t) for t in typeset):
            fields.append(pa.field(name, pa.timestamp("us")))
        elif any(pa.types.is_boolean(t) for t in typeset):
            fields.append(pa.field(name, pa.bool_()))
        else:
            fields.append(pa.field(name, pa.string()))
    return pa.schema(fields)


def _align_table_to_schema(tbl: pa.Table, schema: pa.Schema) -> pa.Table:
    """Cast / null-fill a table so it matches *schema* exactly."""
    arrays = []
    for field in schema:
        name = field.name
        if name in tbl.column_names:
            col = tbl[name]
            if not col.type.equals(field.type):
                col = pc.cast(col, field.type, safe=False)
            arrays.append(col)
        else:
            arrays.append(pa.nulls(tbl.num_rows, type=field.type))
    return pa.Table.from_arrays(arrays, schema=schema)


def merge_parquets_stream_to_one(
    files: list[Path],
    out_path: Path,
    *,
    batch_rows: int = 100_000,
    row_group_size: int = 100_000,
    compression: str = "zstd",
    overwrite: bool = False,
) -> Path:
    """Stream-merge a list of parquet files into a single output file."""
    if not files:
        raise ValueError("No input parquet files provided for merge.")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and not overwrite:
        print(f"Skipping merge : output already exists: {out_path}")
        return out_path

    target_schema = _build_target_schema(files)
    print(f"Merge target schema: {len(target_schema)} fields")

    writer = pq.ParquetWriter(
        str(out_path),
        target_schema,
        compression=compression,
        use_dictionary=True,
        write_statistics=True,
    )
    total_rows = 0
    try:
        for fi, f in enumerate(files, start=1):
            pf = pq.ParquetFile(str(f))
            print(f"[{fi}/{len(files)}] {f.name}  ({pf.metadata.num_rows:,} rows)")
            for batch in pf.iter_batches(batch_size=batch_rows):
                tbl = pa.Table.from_batches([batch])
                tbl = _align_table_to_schema(tbl, target_schema)
                writer.write_table(tbl, row_group_size=row_group_size)
                total_rows += tbl.num_rows
                del batch, tbl
                gc.collect()
        print(f"Merge complete : {total_rows:,} rows written → {out_path}")
    finally:
        writer.close()
    return out_path


merged_wide_path = merge_parquets_stream_to_one(
    wide_files,
    MERGED_WIDE_PATH,
    batch_rows=MERGE_BATCH_ROWS,
    row_group_size=MERGE_ROW_GROUP_SIZE,
    compression=MERGE_COMPRESSION,
    overwrite=OVERWRITE_MERGED,
)
print("Merged wide parquet:", merged_wide_path)

## 5) Populate / update the DuckDB embeddings cache

Reads the **individual wide parquet files** (not the merged parquet) in Arrow batches and calls `compute_embeddings()` on each batch.  The function skips sample IDs that are already present in the cache for this model hash, so re-running is safe and incremental.

Key points:
- All batches within a file are iterated — previously a `next()` on the iterator silently dropped everything beyond the first `PARQUET_BATCH_ROWS` rows.
- `FORCE_RECOMPUTE = True` deletes existing cache entries for all samples before inserting fresh ones.
- Loguru is set to `WARNING` level to suppress the per-batch debug output.

In [ ]:

# Suppress debug/info noise from the embeddings cache internals
logger.remove()
logger.add(sys.stderr, level="WARNING")

if not WIDE_DIR.exists():
    raise FileNotFoundError(f"Wide directory not found: {WIDE_DIR}")

wide_files_for_cache = sorted(WIDE_DIR.glob(f"*{WIDE_SUFFIX}.parquet"))
if not wide_files_for_cache:
    raise RuntimeError(f"No wide parquet files found under: {WIDE_DIR}")
print(f"Found {len(wide_files_for_cache)} wide parquet files to embed.")

torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {torch_device}")

PRESTO_URL = "/projects/worldcereal/models/roshaan/Global/WorldCerealPresto-NoSpatialGlobalLC10CT23-NewDataRegions-month-augment=True-balance=TruePerBin-SpatialBin5.0deg-timeexplicit=True-masking=enabled-ema0.2-clamp=0.2-8.0-run=202606040738/WorldCerealPresto-NoSpatialGlobalLC10CT23-NewDataRegions-month-augment=True-balance=TruePerBin-SpatialBin5.0deg-timeexplicit=True-masking=enabled-ema0.2-clamp=0.2-8.0-run=202606040738_encoder.pt"

# WHY NOT Presto(pretrained_model_path=PRESTO_URL):
# This model was trained with time_explicit=True, starting from posembed72.pt as
# the backbone. Its encoder checkpoint therefore has pos_embed [1, 72, 64].
# But Presto(pretrained_model_path=...) first builds a fresh base model with
# pos_embed [1, 24, 64] and THEN tries to load the checkpoint — causing a size
# mismatch before the wrapper has a chance to resize to 72.
#
# Fix: create Presto() with NO pretrained_model_path (the constructor's own
# __init__ always resizes pos_embed to 72 regardless), then load the fine-tuned
# encoder weights manually. At that point both model and checkpoint have shape
# [1, 72, 64] and the load succeeds.
model = Presto()  # pos_embed auto-resized to 72 inside __init__
state_dict = torch.load(PRESTO_URL, map_location=torch_device)
missing, unexpected = model.load_state_dict(state_dict, strict=False)
if missing:
    print(f"Missing keys (expected if head is absent): {missing}")
if unexpected:
    print(f"Unexpected keys: {unexpected}")
model.eval().to(torch_device)

model_hash = get_model_hash(model)
print(f"Model hash: {model_hash}")

# Reduce Batch size here to avoid OOM errors during embedding computation
BATCH_SIZE = 4096

for wide_file in tqdm(wide_files_for_cache, desc="Embedding files", unit="file"):
    pf = pq.ParquetFile(str(wide_file))
    total_rows = pf.metadata.num_rows

    # Iterate ALL Arrow batches in the file.
    # (Using next() on the iterator would silently drop everything beyond
    # the first PARQUET_BATCH_ROWS rows.)
    rows_processed = 0
    for batch in pf.iter_batches(batch_size=PARQUET_BATCH_ROWS):
        tbl = pa.Table.from_batches([batch])
        df = tbl.to_pandas()
        rows_processed += len(df)
        compute_embeddings(
            df,
            model=model,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            embeddings_db_path=str(EMBEDDINGS_DB_PATH),
            force_recompute=FORCE_RECOMPUTE,
            show_progress=True,
        )
        del batch, tbl, df
        gc.collect()

print("Embeddings cache population complete.")


In [ ]:
# -----------------------------------------------------------------------
# ALTERNATIVE: Run embeddings from the merged wide parquet (if RAM permits)
#
# Reading everything at once is faster (single pass) but requires enough
# memory to hold the full merged wide parquet.  For very large datasets
# prefer the file-by-file approach above.
# -----------------------------------------------------------------------

# df_merged = pd.read_parquet(str(MERGED_WIDE_PATH))
# compute_embeddings(
#     df_merged,
#     model=model,
#     batch_size=BATCH_SIZE,
#     num_workers=NUM_WORKERS,
#     embeddings_db_path=str(EMBEDDINGS_DB_PATH),
#     force_recompute=FORCE_RECOMPUTE,
#     show_progress=True,
# )

## 6) Load class mappings from SharePoint

The outlier pipeline needs a mapping from raw `ewoc_code` integers to human-readable class names (e.g. `LANDCOVER10`, `CROPTYPE24`).  The mapping is maintained in an Excel file on SharePoint and fetched via the Microsoft Graph API.

Credentials are loaded from a `.sharepointenv` file (not committed to git).  The variables `WORLDCEREAL_SP_SITE_URL` and `WORLDCEREAL_SP_FILE_URL` must be set there.

The resulting `CLASS_MAPPINGS` dict is keyed by mapping name (e.g. `"LANDCOVER10"`, `"CROPTYPE24"`) and will be passed to `run_pipeline` below.

In [ ]:
import json
import os
import pandas as pd

# ── Smoke-test path: load from local class_mappings.json (no SharePoint needed) ──
if "SMOKE_CLASS_MAPPINGS_JSON" in dir() and SMOKE_CLASS_MAPPINGS_JSON is not None:
    print(f"[SMOKE TEST] Loading class mappings from: {SMOKE_CLASS_MAPPINGS_JSON}")
    with open(SMOKE_CLASS_MAPPINGS_JSON) as _f:
        CLASS_MAPPINGS = json.load(_f)
    print("CLASS_MAPPINGS keys:", list(CLASS_MAPPINGS.keys()))

# ── Production path: fetch from SharePoint ─────────────────────────────────────
else:
    from dotenv import load_dotenv
    from worldcereal.utils.sharepoint import (
        get_excel_from_sharepoint,
        build_class_mappings,
    )

    _env_candidates = [
        Path("~/.sharepointenv"),
        Path("/path/to/TestFolder/.sharepointenv"),
    ]
    _env_path = next((p for p in _env_candidates if p.exists()), None)
    assert _env_path is not None, (
        f".sharepointenv not found at any of: {[str(p) for p in _env_candidates]}"
    )
    print(f"Using .sharepointenv: {_env_path}")
    load_dotenv(_env_path, override=True)

    SITE_URL = os.environ["WORLDCEREAL_SP_SITE_URL"]
    FILE_URL  = os.environ["WORLDCEREAL_SP_FILE_URL"]

    legend = get_excel_from_sharepoint(
        site_url=SITE_URL,
        file_server_relative_url=FILE_URL,
        retries=10,
        sheet_name=0,
    )
    print("Successfully fetched legend from SharePoint")

    legend["ewoc_code"] = (
        legend["ewoc_code"]
        .astype("string")
        .str.replace("-", "", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
        .astype("Int64")
    )

    CLASS_MAPPINGS = build_class_mappings(legend)
    print("CLASS_MAPPINGS keys:", list(CLASS_MAPPINGS.keys()))


## 7) Scoring : RERUN mode

> **Skip this section if `PIPELINE_MODE = "update"`** — jump to section 7u below instead.

Scores every sample in the DuckDB embeddings cache against its spatial neighbours.  This loads the **full** cache and is intended for HPC with plenty of RAM.

### 7a) LANDCOVER10 (rerun)

Key parameters:
- `h3_level = [2, 3]` — adaptive: coarser level for sparse regions, finer for dense ones
- `max_slice_size = 10 000` — caps how large a slice can be before it's pushed to a finer H3 level
- `mad_k = 4.0` — number of MADs above the median to flag as an outlier
- `skip_classes = ["ignore"]` — samples mapped to the "ignore" class are excluded from scoring


In [ ]:
# sleep for 3 hours
import time

time.sleep(3.5 * 60 * 60)  # 3.5 hours in seconds

In [ ]:
LC10_flagged_gdf, LC10_summary_df = run_pipeline(
    embeddings_db_path=str(EMBEDDINGS_DB_PATH),
    restrict_model_hash=None,
    label_domain="LANDCOVER10",
    map_to_finetune=False,
    class_mappings_name="LANDCOVER10",
    skip_classes=["ignore"],
    mapping_file=CLASS_MAPPINGS,
    h3_level=[2, 3],            # adaptive: coarse → fine
    # -- Slice definition ---------------------------------------------
    # group_cols joins the slice key on top of the H3 cell and the label.
    #   []          (DEFAULT)  pool across source datasets, so a sample is compared
    #               with the same class in the same locality whoever digitised it.
    #               That comparison is the assumption the whole method rests on.
    #               It also avoids the uneven-dataset-size trap: split by ref_id, a
    #               1.2k-point dataset fragments below min_slice_size and comes back
    #               `unscored` while the 25k one next door is fully scrutinised.
    #   ["ref_id"]  score each dataset against itself. Note a wholly-mislabelled
    #               dataset then forms its own self-consistent slice and becomes
    #               undetectable (measured: abs_z -0.15, 0/360 flagged, vs abs_z
    #               5.45 and 339/360 when pooled).
    group_cols=[],
    min_slice_size=200,
    max_slice_size=10_000,
    merge_small_slice=True,
    max_merge_iterations=16,
    threshold_mode="stable_mad",   # local median, cross-slice sigma
    percentile_q=0.96,
    mad_k=3.3,
    abs_threshold=None,
    fdr_alpha=0.05,
    min_flagged_per_slice=None,
    max_flagged_fraction=None,
    max_full_pairwise_n=0,      # disable full pairwise matrix (expensive)
    norm_percentiles=(2.0, 98.0),
    # -- Absolute gate (see README, "How a sample gets flagged") ------
    # A flag now needs BOTH a within-slice signal and an absolute one, measured
    # against a null pooled across many slices of the same class.  Without it the
    # top ~2% of EVERY slice clears the escalation thresholds whether or not the
    # slice holds a single real error.  Set require_absolute=False to reproduce
    # the old relative-only behaviour for an ablation.
    require_absolute=True,
    abs_z_k=3.3,                # robust sigma required to flag
    abs_z_suspect=4.0,
    abs_z_candidate=5.5,
    abs_combine="min",          # demand centroid AND neighbourhood evidence
    # -- Null localisation --------------------------------------------
    # The null answers "how far from its own slice centroid does a typical sample
    # of this class sit?". It is conditioned on TWO things, in this order:
    #
    #   h3_null_region  WHERE the slice is. Pooling globally is wrong: wheat in a
    #                   uniform monoculture disperses far less around its local
    #                   centroid than wheat in a fragmented smallholder landscape,
    #                   so a global null is set by whichever landscape contributes
    #                   most slices and every more-variable region then looks
    #                   anomalous as a whole. The region is RELATIVE to the slice's
    #                   own resolution: max(slice_res - offset, min_level).
    #   h3_null_res     AT WHAT SCALE it was sliced. A distance distribution scales
    #                   with cell size -- L2 spans 49x the area of L4 -- so slices
    #                   at different resolutions must not share a null. The region
    #                   key alone does not separate them: floored at L1, one region
    #                   holds both L2 and L3 slices. This is why a 3-level croptype
    #                   run ([2,3,4]) behaved worse than a 2-level landcover run.
    #
    # The keys are a NESTING, coarsest first. A null is estimated at every prefix
    # -- (class), (class,region), (class,region,res) -- and each row uses the
    # FINEST group that exists for it, each depth shrunk toward its own parent by
    # w = n_slices/(n_slices+null_shrink_k). That backing-off is what makes the
    # extra key safe: without it a thin (class,region,res) group fell all the way
    # back to the flat global null, which is exactly the mixing the key removes,
    # for exactly the rare-class-in-a-fine-cell case that needs it.
    #
    # Measured end to end on a co-located L2/L3/L4 croptype run, 15% planted:
    #     class only                        recall 30.9%  clean FP 0.368%
    #     class + fixed L1 region           recall 30.9%  clean FP 0.368%
    #     class + relative region           recall 37.4%  clean FP 0.150%
    #     class + region + resolution       recall 38.5%  clean FP 0.109%
    # The two keys fix different levels. The relative region rescues L4 (recall
    # 32% -> 71%): an L4 slice's region is an L2 cell holding only other L4
    # slices, so the region separates resolutions there by itself. The
    # resolution key does the rest -- L2 and L3 share one L1 region, and
    # separating them lifts L3 recall 25% -> 31% and halves L2's false
    # positives 0.37% -> 0.20%. For a 2-level LANDCOVER10 run both levels floor
    # to the same L1 region, so only the resolution key bites: recall flat
    # (24.8% -> 25.0%) at half the false positives (0.252% -> 0.112%).
    #
    # The run prints a depth histogram; a large share at depth 0-1 means the keys
    # are too fine for this collection's density (raise null_region_offset).
    # Set null_extra_keys=[] to pool globally, or ["h3_null_region"] to drop only
    # the resolution key. null_region_level pins an ABSOLUTE region resolution
    # instead (legacy, and wrong for any multi-resolution run).
    null_extra_keys=["h3_null_region", "h3_null_res"],
    null_region_offset=2,
    null_region_min_level=1,
    null_region_level=None,
    null_shrink_k=5.0,
    # -- Heavy slice contamination ------------------------------------
    # The scale, not the centroid, was the blocker: the cross-slice null took
    # the median of per-slice MADs, and a MAD is widened by the very right-side
    # errors it measures against -- so when EVERY slice of a class carries
    # 30-45% errors the null inflated with them and nothing cleared the gate.
    # "left_tail" measures the spread as median-q25, i.e. from the clean left
    # half only. Measured recall at 40% slice contamination: 0.14 -> 0.90, at a
    # slightly LOWER clean false-positive rate. Use "mad" for ablations only.
    null_scale_estimator="left_tail",
    purity_veto=0.80,           # cap escalation when the neighbours agree
    # -- Coverage / quality -------------------------------------------
    min_scoring_slice_size=50,  # below this -> flag "unscored", not "normal"
    quality_gate=True,          # quarantine degenerate embeddings as "unscorable"
    strict_quality=False,       # True = hard-error on mixed model_hash / bad H3
    # -- Temporal control ---------------------------------------------
    # Embeddings are season-specific and the collection spans many years, so a
    # sample from a minority year is distant for phenological, not label, reasons.
    # Point this at a year/season column IF the embeddings carry one -- the
    # standard cache schema does not, and the run will say so if it cannot.
    time_col=None,
    output_samples_path=LC10_SAMPLES_PATH,
    output_summary_path=LC10_SUMMARY_PATH,
    debug=False,
    centroid_mode="trimmed", centroid_trim=0.45,  # >= expected contamination
    gate_confidence_by_flag=True,      # set False to keep old continuous-everywhere confidence
    apply_slice_trust=False, slice_trust_min=0.05,
)
print(f"LC10 pipeline done — {len(LC10_flagged_gdf):,} samples scored.")

In [ ]:
# Read back from disk (useful if resuming the notebook without re-running the pipeline)
import pandas as pd

_lc10_cols = ["ref_id", "sample_id", "LANDCOVER10", "confidence_nonoutlier", "anomaly_flag"]
LC10_flagged_gdf = pd.read_parquet(LC10_SAMPLES_PATH, columns=_lc10_cols)


In [ ]:
# Keep only the anomaly columns relevant for downstream merging and rename for clarity
LC10_flagged_gdf = LC10_flagged_gdf[
    ["ref_id", "sample_id", "LANDCOVER10", "confidence_nonoutlier", "anomaly_flag"]
]
LC10_flagged_gdf = LC10_flagged_gdf.rename(columns={
    "confidence_nonoutlier": "LC10_confidence_nonoutlier",
    "anomaly_flag":          "LC10_anomaly_flag",
    "LANDCOVER10":           "outlier_LC10_cls",
})
print(f"LC10 scores: {len(LC10_flagged_gdf):,} rows")
LC10_flagged_gdf.head()

### 7b) CROPTYPE24 (rerun)

Same approach as LC10 but using the finer `CROPTYPE24` label schema (24 crop-type classes).  Because crop-type classes are more numerous and individual classes are sparser, we use three H3 levels (`[2, 3, 4]`) with a smaller per-slice cap (`max_slice_size = 5 000`) to keep slices manageable.


In [ ]:
CTY24_flagged_gdf, CTY24_summary_df = run_pipeline(
    embeddings_db_path=str(EMBEDDINGS_DB_PATH),
    restrict_model_hash=None,
    label_domain="CROPTYPE24",
    map_to_finetune=False,
    class_mappings_name="CROPTYPE24",
    skip_classes=["ignore"],
    mapping_file=CLASS_MAPPINGS,
    h3_level=[2, 3, 4],         # adaptive: coarse → fine (3 levels for sparse crop types)
    # -- Slice definition ---------------------------------------------
    # group_cols joins the slice key on top of the H3 cell and the label.
    #   []          (DEFAULT)  pool across source datasets, so a sample is compared
    #               with the same class in the same locality whoever digitised it.
    #               That comparison is the assumption the whole method rests on.
    #               It also avoids the uneven-dataset-size trap: split by ref_id, a
    #               1.2k-point dataset fragments below min_slice_size and comes back
    #               `unscored` while the 25k one next door is fully scrutinised.
    #   ["ref_id"]  score each dataset against itself. Note a wholly-mislabelled
    #               dataset then forms its own self-consistent slice and becomes
    #               undetectable (measured: abs_z -0.15, 0/360 flagged, vs abs_z
    #               5.45 and 339/360 when pooled).
    group_cols=[],
    min_slice_size=100,
    max_slice_size=5_000,
    merge_small_slice=True,
    max_merge_iterations=8,
    threshold_mode="stable_mad",   # local median, cross-slice sigma
    percentile_q=0.96,
    mad_k=3.3,
    abs_threshold=None,
    fdr_alpha=0.05,
    min_flagged_per_slice=None,
    max_flagged_fraction=None,
    max_full_pairwise_n=0,      # disable full pairwise matrix (expensive)
    norm_percentiles=(2.0, 98.0),
    # -- Absolute gate (see README, "How a sample gets flagged") ------
    # A flag now needs BOTH a within-slice signal and an absolute one, measured
    # against a null pooled across many slices of the same class.  Without it the
    # top ~2% of EVERY slice clears the escalation thresholds whether or not the
    # slice holds a single real error.  Set require_absolute=False to reproduce
    # the old relative-only behaviour for an ablation.
    require_absolute=True,
    abs_z_k=3.3,                # robust sigma required to flag
    abs_z_suspect=4.0,
    abs_z_candidate=5.5,
    abs_combine="min",          # demand centroid AND neighbourhood evidence
    # -- Null localisation --------------------------------------------
    # The null answers "how far from its own slice centroid does a typical sample
    # of this class sit?". It is conditioned on TWO things, in this order:
    #
    #   h3_null_region  WHERE the slice is. Pooling globally is wrong: wheat in a
    #                   uniform monoculture disperses far less around its local
    #                   centroid than wheat in a fragmented smallholder landscape,
    #                   so a global null is set by whichever landscape contributes
    #                   most slices and every more-variable region then looks
    #                   anomalous as a whole. The region is RELATIVE to the slice's
    #                   own resolution: max(slice_res - offset, min_level).
    #   h3_null_res     AT WHAT SCALE it was sliced. A distance distribution scales
    #                   with cell size -- L2 spans 49x the area of L4 -- so slices
    #                   at different resolutions must not share a null. The region
    #                   key alone does not separate them: floored at L1, one region
    #                   holds both L2 and L3 slices. This is why a 3-level croptype
    #                   run ([2,3,4]) behaved worse than a 2-level landcover run.
    #
    # The keys are a NESTING, coarsest first. A null is estimated at every prefix
    # -- (class), (class,region), (class,region,res) -- and each row uses the
    # FINEST group that exists for it, each depth shrunk toward its own parent by
    # w = n_slices/(n_slices+null_shrink_k). That backing-off is what makes the
    # extra key safe: without it a thin (class,region,res) group fell all the way
    # back to the flat global null, which is exactly the mixing the key removes,
    # for exactly the rare-class-in-a-fine-cell case that needs it.
    #
    # Measured end to end on a co-located L2/L3/L4 croptype run, 15% planted:
    #     class only                        recall 30.9%  clean FP 0.368%
    #     class + fixed L1 region           recall 30.9%  clean FP 0.368%
    #     class + relative region           recall 37.4%  clean FP 0.150%
    #     class + region + resolution       recall 38.5%  clean FP 0.109%
    # The two keys fix different levels. The relative region rescues L4 (recall
    # 32% -> 71%): an L4 slice's region is an L2 cell holding only other L4
    # slices, so the region separates resolutions there by itself. The
    # resolution key does the rest -- L2 and L3 share one L1 region, and
    # separating them lifts L3 recall 25% -> 31% and halves L2's false
    # positives 0.37% -> 0.20%. For a 2-level LANDCOVER10 run both levels floor
    # to the same L1 region, so only the resolution key bites: recall flat
    # (24.8% -> 25.0%) at half the false positives (0.252% -> 0.112%).
    #
    # The run prints a depth histogram; a large share at depth 0-1 means the keys
    # are too fine for this collection's density (raise null_region_offset).
    # Set null_extra_keys=[] to pool globally, or ["h3_null_region"] to drop only
    # the resolution key. null_region_level pins an ABSOLUTE region resolution
    # instead (legacy, and wrong for any multi-resolution run).
    null_extra_keys=["h3_null_region", "h3_null_res"],
    null_region_offset=2,
    null_region_min_level=1,
    null_region_level=None,
    null_shrink_k=5.0,
    # -- Heavy slice contamination ------------------------------------
    # The scale, not the centroid, was the blocker: the cross-slice null took
    # the median of per-slice MADs, and a MAD is widened by the very right-side
    # errors it measures against -- so when EVERY slice of a class carries
    # 30-45% errors the null inflated with them and nothing cleared the gate.
    # "left_tail" measures the spread as median-q25, i.e. from the clean left
    # half only. Measured recall at 40% slice contamination: 0.14 -> 0.90, at a
    # slightly LOWER clean false-positive rate. Use "mad" for ablations only.
    null_scale_estimator="left_tail",
    purity_veto=0.80,           # cap escalation when the neighbours agree
    # -- Coverage / quality -------------------------------------------
    min_scoring_slice_size=50,  # below this -> flag "unscored", not "normal"
    quality_gate=True,          # quarantine degenerate embeddings as "unscorable"
    strict_quality=False,       # True = hard-error on mixed model_hash / bad H3
    # -- Temporal control ---------------------------------------------
    # Embeddings are season-specific and the collection spans many years, so a
    # sample from a minority year is distant for phenological, not label, reasons.
    # Point this at a year/season column IF the embeddings carry one -- the
    # standard cache schema does not, and the run will say so if it cannot.
    time_col=None,
    output_samples_path=CTY24_SAMPLES_PATH,
    output_summary_path=CTY24_SUMMARY_PATH,
    debug=False,
    centroid_mode="trimmed", centroid_trim=0.45,  # >= expected contamination
    gate_confidence_by_flag=True,      # set False to keep old continuous-everywhere confidence
    apply_slice_trust=False, slice_trust_min=0.05,
)
print(f"CTY24 pipeline done — {len(CTY24_flagged_gdf):,} samples scored.")

In [ ]:
# Read back from disk (useful if resuming the notebook without re-running the pipeline)
import pandas as pd

_cty24_cols = ["ref_id", "sample_id", "CROPTYPE24", "confidence_nonoutlier", "anomaly_flag"]
CTY24_flagged_gdf = pd.read_parquet(CTY24_SAMPLES_PATH, columns=_cty24_cols)


In [ ]:
# Keep only the anomaly columns relevant for downstream merging and rename for clarity
CTY24_flagged_gdf = CTY24_flagged_gdf[
    ["ref_id", "sample_id", "CROPTYPE24", "confidence_nonoutlier", "anomaly_flag"]
]
CTY24_flagged_gdf = CTY24_flagged_gdf.rename(columns={
    "confidence_nonoutlier": "CTY24_confidence_nonoutlier",
    "anomaly_flag":          "CTY24_anomaly_flag",
    "CROPTYPE24":            "outlier_CTY24_cls",
})
print(f"CTY24 scores: {len(CTY24_flagged_gdf):,} rows")
CTY24_flagged_gdf.head()

# In rerun mode: all files are rewritten during write-back (no ref_id filtering)
RESCORED_REF_IDS = None


## 7u) Scoring : UPDATE mode (incremental)

> **Skip this section if `PIPELINE_MODE = "rerun"`** — use section 7 above instead.

Instead of loading the entire DuckDB embeddings cache, the update mode:

1. **Per domain** (LC10, CTY24), scans the long-format parquets for rows with NaN in that domain's anomaly columns only.
2. Excludes "ignore"-class samples (they are *always* NaN by design — `run_pipeline` holds them aside).
3. Computes the **H3 impact zone** (unscored cells + neighbours).
4. Loads **only** the embeddings that fall inside the impact zone from DuckDB (memory-efficient two-phase query).
5. Scores each domain separately, then outer-merges.

This is dramatically faster and uses far less memory than a full rerun — ideal for VMs.

### Helper: resolve skip-class ewoc_codes

Resolves which `ewoc_code` values map to a skip-class label (e.g. "ignore") for a given domain.  These samples always have NaN anomaly scores by design, so they should be excluded from the "unscored" set before computing the impact zone.


In [ ]:
# ======================================================================
# UPDATE MODE — Incremental scoring (per-domain impact zone)
# ======================================================================
# This cell replaces sections 7a/7b when PIPELINE_MODE == "update".
# It produces the same LC10_flagged_gdf and CTY24_flagged_gdf DataFrames
# that the rerun cells produce, so sections 8–10 work identically.
# ======================================================================

if PIPELINE_MODE != "update":
    print(f"[7u] Skipping update cell : PIPELINE_MODE='{PIPELINE_MODE}' (not 'update').")
    # RESCORED_REF_IDS is already set to None by the rerun cells (7a/7b)
else:
    # ------------------------------------------------------------------
    # Helper: resolve skip-class ewoc_codes for a domain
    # ------------------------------------------------------------------
    def _resolve_skip_ewoc_codes(class_mappings, class_mappings_name, skip_classes):
        """Return set of ewoc_code strings that map to a skip_class label."""
        if not skip_classes:
            return set()
        mapping = class_mappings.get(class_mappings_name)
        if not isinstance(mapping, dict):
            return set()
        skip_set = {str(s).lower() for s in skip_classes}
        return {
            str(ewoc_code)
            for ewoc_code, label in mapping.items()
            if str(label).lower() in skip_set
        }

    # ------------------------------------------------------------------
    # Helper: per-domain unscored detection → impact zone → embeddings load
    # ------------------------------------------------------------------
    def _discover_domain_impact(
        *,
        domain_label,
        domain_anomaly_cols,
        h3_levels,
        long_parquet_dir,
        embeddings_db_path,
        neighbour_rings,
        skip_ewoc_codes=None,
        parquet_glob="*.geoparquet",
    ):
        """Detect unscored samples for one domain, compute impact zone, load embeddings.

        Returns (affected_df, embed_cols, rescored_ref_ids) or (None, None, None).
        """
        print(f"[update/{domain_label}] Scanning for unscored samples "
              f"(checking {domain_anomaly_cols}) ...")
        # flag_col makes the "already handled?" test decision-based: a row with a
        # terminal flag (including unscored / unscorable / unmapped / skipped)
        # counts as done even though some numeric columns are legitimately null.
        # The old "any column is NaN" rule rediscovered those rows on EVERY
        # update run, so the impact zone grew and the mode never converged.
        _flag_cols = [c for c in domain_anomaly_cols if c.endswith("_anomaly_flag")]
        unscored = find_unscored_samples(
            long_parquet_dir=long_parquet_dir,
            anomaly_cols=domain_anomaly_cols,
            parquet_glob=parquet_glob,
            flag_col=_flag_cols[0] if _flag_cols else None,
        )
        if unscored.empty:
            print(f"[update/{domain_label}] No unscored samples — nothing to do.")
            return None, None, None

        print(f"[update/{domain_label}] {len(unscored):,} unscored (ref_id, sample_id) pairs")

        # Look up H3 cells + ewoc_code for skip-class filtering
        print(f"[update/{domain_label}] Looking up H3 cells for unscored samples ...")
        con = duckdb.connect(str(embeddings_db_path), read_only=True)
        try:
            ids_df = unscored[["sample_id"]].drop_duplicates()
            con.register("unscored_ids", ids_df)
            h3_df = con.execute(
                "SELECT e.sample_id, e.h3_l3_cell, e.ewoc_code "
                "FROM embeddings_cache e "
                "INNER JOIN unscored_ids u ON e.sample_id = u.sample_id"
            ).fetchdf()
        finally:
            con.close()

        # Filter out skip-class ewoc_codes
        if skip_ewoc_codes and not h3_df.empty:
            before = len(h3_df)
            h3_df["_ewoc_str"] = h3_df["ewoc_code"].astype(str)
            h3_df = h3_df[~h3_df["_ewoc_str"].isin(skip_ewoc_codes)].drop(columns=["_ewoc_str"])
            after = len(h3_df)
            print(f"[update/{domain_label}] Filtered {before - after:,} skip-class rows "
                  f"→ {after:,} remain")

        if h3_df.empty:
            print(f"[update/{domain_label}] All unscored are skip-class — nothing to do.")
            return None, None, None

        print(f"[update/{domain_label}] {len(h3_df):,} unscored samples found in cache")
        h3_df = h3_df[["sample_id", "h3_l3_cell"]]

        unscored_h3_cells = h3_df["h3_l3_cell"].dropna().unique().tolist()
        impact_cells = compute_impact_zone(
            unscored_h3_cells=unscored_h3_cells,
            h3_levels=h3_levels,
            neighbour_rings=neighbour_rings,
        )
        print(f"[update/{domain_label}] Impact zone: {len(impact_cells):,} H3 cells")

        print(f"[update/{domain_label}] Loading impact-zone embeddings from cache ...")
        affected_df, embed_cols = load_affected_embeddings_from_cache(
            embeddings_db_path=str(embeddings_db_path),
            impact_cells=impact_cells,
            h3_levels=h3_levels,
        )
        if affected_df.empty:
            print(f"[update/{domain_label}] No embeddings matched impact zone.")
            return None, None, None

        rescored_ref_ids = set(affected_df["ref_id"].astype(str).unique())
        print(f"[update/{domain_label}] Re-scoring {len(affected_df):,} samples "
              f"across {len(rescored_ref_ids):,} ref_ids")
        return affected_df, embed_cols, rescored_ref_ids

    # ------------------------------------------------------------------
    # Resolve skip-class ewoc_codes
    # ------------------------------------------------------------------
    SKIP_CLASSES = ["ignore"]

    lc10_skip_codes = _resolve_skip_ewoc_codes(CLASS_MAPPINGS, "LANDCOVER10", SKIP_CLASSES)
    cty24_skip_codes = _resolve_skip_ewoc_codes(CLASS_MAPPINGS, "CROPTYPE24", SKIP_CLASSES)
    print(f"Skip-class ewoc_codes — LC10: {len(lc10_skip_codes):,}, CTY24: {len(cty24_skip_codes):,}")

    # ------------------------------------------------------------------
    # LC10 domain
    # ------------------------------------------------------------------
    print("\n=== LANDCOVER10 domain ===")
    lc10_result = _discover_domain_impact(
        domain_label="LC10",
        domain_anomaly_cols=list(LC10_ANOMALY_COLUMNS),
        h3_levels=[2, 3],
        long_parquet_dir=INPUT_LONG_DIR,
        embeddings_db_path=EMBEDDINGS_DB_PATH,
        neighbour_rings=NEIGHBOUR_RINGS,
        skip_ewoc_codes=lc10_skip_codes,
        parquet_glob=PARQUET_GLOB,
    )
    lc10_affected_df, lc10_embed_cols, lc10_ref_ids = lc10_result

    LC10_flagged_gdf = pd.DataFrame()
    if lc10_affected_df is not None:
        print("[update] Running LANDCOVER10 scoring (impact zone) ...")
        LC10_flagged_gdf, _ = run_pipeline(
            embeddings_db_path=str(EMBEDDINGS_DB_PATH),
            restrict_model_hash=None,
            label_domain="LANDCOVER10",
            map_to_finetune=False,
            class_mappings_name="LANDCOVER10",
            skip_classes=SKIP_CLASSES,
            mapping_file=CLASS_MAPPINGS,
            h3_level=[2, 3],
            # -- Slice definition ---------------------------------------------
            # group_cols joins the slice key on top of the H3 cell and the label.
            #   []          (DEFAULT)  pool across source datasets, so a sample is compared
            #               with the same class in the same locality whoever digitised it.
            #               That comparison is the assumption the whole method rests on.
            #               It also avoids the uneven-dataset-size trap: split by ref_id, a
            #               1.2k-point dataset fragments below min_slice_size and comes back
            #               `unscored` while the 25k one next door is fully scrutinised.
            #   ["ref_id"]  score each dataset against itself. Note a wholly-mislabelled
            #               dataset then forms its own self-consistent slice and becomes
            #               undetectable (measured: abs_z -0.15, 0/360 flagged, vs abs_z
            #               5.45 and 339/360 when pooled).
            group_cols=[],
            min_slice_size=200,
            max_slice_size=10_000,
            merge_small_slice=True,
            max_merge_iterations=16,
            threshold_mode="stable_mad",   # local median, cross-slice sigma
            percentile_q=0.96,
            mad_k=3.3,
            abs_threshold=None,
            fdr_alpha=0.05,
            min_flagged_per_slice=None,
            max_flagged_fraction=None,
            max_full_pairwise_n=0,
            norm_percentiles=(2.0, 98.0),
            # -- Absolute gate (see README, "How a sample gets flagged") ------
            # A flag now needs BOTH a within-slice signal and an absolute one, measured
            # against a null pooled across many slices of the same class.  Without it the
            # top ~2% of EVERY slice clears the escalation thresholds whether or not the
            # slice holds a single real error.  Set require_absolute=False to reproduce
            # the old relative-only behaviour for an ablation.
            require_absolute=True,
            abs_z_k=3.3,                # robust sigma required to flag
            abs_z_suspect=4.0,
            abs_z_candidate=5.5,
            abs_combine="min",          # demand centroid AND neighbourhood evidence
    # -- Null localisation --------------------------------------------
    # The null answers "how far from its own slice centroid does a typical sample
    # of this class sit?". It is conditioned on TWO things, in this order:
    #
    #   h3_null_region  WHERE the slice is. Pooling globally is wrong: wheat in a
    #                   uniform monoculture disperses far less around its local
    #                   centroid than wheat in a fragmented smallholder landscape,
    #                   so a global null is set by whichever landscape contributes
    #                   most slices and every more-variable region then looks
    #                   anomalous as a whole. The region is RELATIVE to the slice's
    #                   own resolution: max(slice_res - offset, min_level).
    #   h3_null_res     AT WHAT SCALE it was sliced. A distance distribution scales
    #                   with cell size -- L2 spans 49x the area of L4 -- so slices
    #                   at different resolutions must not share a null. The region
    #                   key alone does not separate them: floored at L1, one region
    #                   holds both L2 and L3 slices. This is why a 3-level croptype
    #                   run ([2,3,4]) behaved worse than a 2-level landcover run.
    #
    # The keys are a NESTING, coarsest first. A null is estimated at every prefix
    # -- (class), (class,region), (class,region,res) -- and each row uses the
    # FINEST group that exists for it, each depth shrunk toward its own parent by
    # w = n_slices/(n_slices+null_shrink_k). That backing-off is what makes the
    # extra key safe: without it a thin (class,region,res) group fell all the way
    # back to the flat global null, which is exactly the mixing the key removes,
    # for exactly the rare-class-in-a-fine-cell case that needs it.
    #
    # Measured end to end on a co-located L2/L3/L4 croptype run, 15% planted:
    #     class only                        recall 30.9%  clean FP 0.368%
    #     class + fixed L1 region           recall 30.9%  clean FP 0.368%
    #     class + relative region           recall 37.4%  clean FP 0.150%
    #     class + region + resolution       recall 38.5%  clean FP 0.109%
    # The two keys fix different levels. The relative region rescues L4 (recall
    # 32% -> 71%): an L4 slice's region is an L2 cell holding only other L4
    # slices, so the region separates resolutions there by itself. The
    # resolution key does the rest -- L2 and L3 share one L1 region, and
    # separating them lifts L3 recall 25% -> 31% and halves L2's false
    # positives 0.37% -> 0.20%. For a 2-level LANDCOVER10 run both levels floor
    # to the same L1 region, so only the resolution key bites: recall flat
    # (24.8% -> 25.0%) at half the false positives (0.252% -> 0.112%).
    #
    # The run prints a depth histogram; a large share at depth 0-1 means the keys
    # are too fine for this collection's density (raise null_region_offset).
    # Set null_extra_keys=[] to pool globally, or ["h3_null_region"] to drop only
    # the resolution key. null_region_level pins an ABSOLUTE region resolution
    # instead (legacy, and wrong for any multi-resolution run).
    null_extra_keys=["h3_null_region", "h3_null_res"],
    null_region_offset=2,
    null_region_min_level=1,
    null_region_level=None,
    null_shrink_k=5.0,
            # -- Heavy slice contamination ------------------------------------
            # The scale, not the centroid, was the blocker: the cross-slice null took
            # the median of per-slice MADs, and a MAD is widened by the very right-side
            # errors it measures against -- so when EVERY slice of a class carries
            # 30-45% errors the null inflated with them and nothing cleared the gate.
            # "left_tail" measures the spread as median-q25, i.e. from the clean left
            # half only. Measured recall at 40% slice contamination: 0.14 -> 0.90, at a
            # slightly LOWER clean false-positive rate. Use "mad" for ablations only.
            null_scale_estimator="left_tail",
            purity_veto=0.80,           # cap escalation when the neighbours agree
            # -- Coverage / quality -------------------------------------------
            min_scoring_slice_size=50,  # below this -> flag "unscored", not "normal"
            quality_gate=True,          # quarantine degenerate embeddings as "unscorable"
            strict_quality=False,       # True = hard-error on mixed model_hash / bad H3
            # -- Temporal control ---------------------------------------------
            # Embeddings are season-specific and the collection spans many years, so a
            # sample from a minority year is distant for phenological, not label, reasons.
            # Point this at a year/season column IF the embeddings carry one -- the
            # standard cache schema does not, and the run will say so if it cannot.
            time_col=None,
            output_samples_path=None,
            output_summary_path=None,
            debug=False,
            embeddings_df=(lc10_affected_df, lc10_embed_cols),
        )
        LC10_flagged_gdf = LC10_flagged_gdf[
            ["ref_id", "sample_id", "LANDCOVER10", "confidence_nonoutlier", "anomaly_flag"]
        ].rename(columns={
            "confidence_nonoutlier": "LC10_confidence_nonoutlier",
            "anomaly_flag": "LC10_anomaly_flag",
            "LANDCOVER10": "outlier_LC10_cls",
        })
        print(f"LC10 update scores: {len(LC10_flagged_gdf):,} rows")
        del lc10_affected_df
        gc.collect()
    else:
        lc10_ref_ids = set()
        print("[update/LC10] Skipped — no unscored samples.")

    # ------------------------------------------------------------------
    # CTY24 domain
    # ------------------------------------------------------------------
    print("\n=== CROPTYPE24 domain ===")
    cty24_result = _discover_domain_impact(
        domain_label="CTY24",
        domain_anomaly_cols=list(CTY24_ANOMALY_COLUMNS),
        h3_levels=[2, 3, 4],
        long_parquet_dir=INPUT_LONG_DIR,
        embeddings_db_path=EMBEDDINGS_DB_PATH,
        neighbour_rings=NEIGHBOUR_RINGS,
        skip_ewoc_codes=cty24_skip_codes,
        parquet_glob=PARQUET_GLOB,
    )
    cty24_affected_df, cty24_embed_cols, cty24_ref_ids = cty24_result

    CTY24_flagged_gdf = pd.DataFrame()
    if cty24_affected_df is not None:
        print("[update] Running CROPTYPE24 scoring (impact zone) ...")
        CTY24_flagged_gdf, _ = run_pipeline(
            embeddings_db_path=str(EMBEDDINGS_DB_PATH),
            restrict_model_hash=None,
            label_domain="CROPTYPE24",
            map_to_finetune=False,
            class_mappings_name="CROPTYPE24",
            skip_classes=SKIP_CLASSES,
            mapping_file=CLASS_MAPPINGS,
            h3_level=[2, 3, 4],
            # -- Slice definition ---------------------------------------------
            # group_cols joins the slice key on top of the H3 cell and the label.
            #   []          (DEFAULT)  pool across source datasets, so a sample is compared
            #               with the same class in the same locality whoever digitised it.
            #               That comparison is the assumption the whole method rests on.
            #               It also avoids the uneven-dataset-size trap: split by ref_id, a
            #               1.2k-point dataset fragments below min_slice_size and comes back
            #               `unscored` while the 25k one next door is fully scrutinised.
            #   ["ref_id"]  score each dataset against itself. Note a wholly-mislabelled
            #               dataset then forms its own self-consistent slice and becomes
            #               undetectable (measured: abs_z -0.15, 0/360 flagged, vs abs_z
            #               5.45 and 339/360 when pooled).
            group_cols=[],
            min_slice_size=100,
            max_slice_size=5_000,
            merge_small_slice=True,
            max_merge_iterations=8,
            threshold_mode="stable_mad",   # local median, cross-slice sigma
            percentile_q=0.96,
            mad_k=3.3,
            abs_threshold=None,
            fdr_alpha=0.05,
            min_flagged_per_slice=None,
            max_flagged_fraction=None,
            max_full_pairwise_n=0,
            norm_percentiles=(2.0, 98.0),
            # -- Absolute gate (see README, "How a sample gets flagged") ------
            # A flag now needs BOTH a within-slice signal and an absolute one, measured
            # against a null pooled across many slices of the same class.  Without it the
            # top ~2% of EVERY slice clears the escalation thresholds whether or not the
            # slice holds a single real error.  Set require_absolute=False to reproduce
            # the old relative-only behaviour for an ablation.
            require_absolute=True,
            abs_z_k=3.3,                # robust sigma required to flag
            abs_z_suspect=4.0,
            abs_z_candidate=5.5,
            abs_combine="min",          # demand centroid AND neighbourhood evidence
    # -- Null localisation --------------------------------------------
    # The null answers "how far from its own slice centroid does a typical sample
    # of this class sit?". It is conditioned on TWO things, in this order:
    #
    #   h3_null_region  WHERE the slice is. Pooling globally is wrong: wheat in a
    #                   uniform monoculture disperses far less around its local
    #                   centroid than wheat in a fragmented smallholder landscape,
    #                   so a global null is set by whichever landscape contributes
    #                   most slices and every more-variable region then looks
    #                   anomalous as a whole. The region is RELATIVE to the slice's
    #                   own resolution: max(slice_res - offset, min_level).
    #   h3_null_res     AT WHAT SCALE it was sliced. A distance distribution scales
    #                   with cell size -- L2 spans 49x the area of L4 -- so slices
    #                   at different resolutions must not share a null. The region
    #                   key alone does not separate them: floored at L1, one region
    #                   holds both L2 and L3 slices. This is why a 3-level croptype
    #                   run ([2,3,4]) behaved worse than a 2-level landcover run.
    #
    # The keys are a NESTING, coarsest first. A null is estimated at every prefix
    # -- (class), (class,region), (class,region,res) -- and each row uses the
    # FINEST group that exists for it, each depth shrunk toward its own parent by
    # w = n_slices/(n_slices+null_shrink_k). That backing-off is what makes the
    # extra key safe: without it a thin (class,region,res) group fell all the way
    # back to the flat global null, which is exactly the mixing the key removes,
    # for exactly the rare-class-in-a-fine-cell case that needs it.
    #
    # Measured end to end on a co-located L2/L3/L4 croptype run, 15% planted:
    #     class only                        recall 30.9%  clean FP 0.368%
    #     class + fixed L1 region           recall 30.9%  clean FP 0.368%
    #     class + relative region           recall 37.4%  clean FP 0.150%
    #     class + region + resolution       recall 38.5%  clean FP 0.109%
    # The two keys fix different levels. The relative region rescues L4 (recall
    # 32% -> 71%): an L4 slice's region is an L2 cell holding only other L4
    # slices, so the region separates resolutions there by itself. The
    # resolution key does the rest -- L2 and L3 share one L1 region, and
    # separating them lifts L3 recall 25% -> 31% and halves L2's false
    # positives 0.37% -> 0.20%. For a 2-level LANDCOVER10 run both levels floor
    # to the same L1 region, so only the resolution key bites: recall flat
    # (24.8% -> 25.0%) at half the false positives (0.252% -> 0.112%).
    #
    # The run prints a depth histogram; a large share at depth 0-1 means the keys
    # are too fine for this collection's density (raise null_region_offset).
    # Set null_extra_keys=[] to pool globally, or ["h3_null_region"] to drop only
    # the resolution key. null_region_level pins an ABSOLUTE region resolution
    # instead (legacy, and wrong for any multi-resolution run).
    null_extra_keys=["h3_null_region", "h3_null_res"],
    null_region_offset=2,
    null_region_min_level=1,
    null_region_level=None,
    null_shrink_k=5.0,
            # -- Heavy slice contamination ------------------------------------
            # The scale, not the centroid, was the blocker: the cross-slice null took
            # the median of per-slice MADs, and a MAD is widened by the very right-side
            # errors it measures against -- so when EVERY slice of a class carries
            # 30-45% errors the null inflated with them and nothing cleared the gate.
            # "left_tail" measures the spread as median-q25, i.e. from the clean left
            # half only. Measured recall at 40% slice contamination: 0.14 -> 0.90, at a
            # slightly LOWER clean false-positive rate. Use "mad" for ablations only.
            null_scale_estimator="left_tail",
            purity_veto=0.80,           # cap escalation when the neighbours agree
            # -- Coverage / quality -------------------------------------------
            min_scoring_slice_size=50,  # below this -> flag "unscored", not "normal"
            quality_gate=True,          # quarantine degenerate embeddings as "unscorable"
            strict_quality=False,       # True = hard-error on mixed model_hash / bad H3
            # -- Temporal control ---------------------------------------------
            # Embeddings are season-specific and the collection spans many years, so a
            # sample from a minority year is distant for phenological, not label, reasons.
            # Point this at a year/season column IF the embeddings carry one -- the
            # standard cache schema does not, and the run will say so if it cannot.
            time_col=None,
            output_samples_path=None,
            output_summary_path=None,
            debug=False,
            embeddings_df=(cty24_affected_df, cty24_embed_cols),
        )
        CTY24_flagged_gdf = CTY24_flagged_gdf[
            ["ref_id", "sample_id", "CROPTYPE24", "confidence_nonoutlier", "anomaly_flag"]
        ].rename(columns={
            "confidence_nonoutlier": "CTY24_confidence_nonoutlier",
            "anomaly_flag": "CTY24_anomaly_flag",
            "CROPTYPE24": "outlier_CTY24_cls",
        })
        print(f"CTY24 update scores: {len(CTY24_flagged_gdf):,} rows")
        del cty24_affected_df
        gc.collect()
    else:
        cty24_ref_ids = set()
        print("[update/CTY24] Skipped — no unscored samples.")

    # Track which ref_ids were rescored (used in write-back to limit file rewrites)
    RESCORED_REF_IDS = (lc10_ref_ids or set()) | (cty24_ref_ids or set())
    print(f"\nTotal rescored ref_ids: {len(RESCORED_REF_IDS):,}")


## 8) Merge LC10 + CTY24 scores

Outer-join the two score tables on `(ref_id, sample_id)` to produce a single DataFrame with one row per sample and six anomaly columns:

| Column | Description |
|--------|-------------|
| `LC10_confidence_nonoutlier` | Confidence score (0–1) that the sample is **not** a LC10 outlier; lower → more suspicious |
| `LC10_anomaly_flag` | Escalation category under LC10: `normal / flagged / suspect / candidate` |
| `outlier_LC10_cls` | Land-cover class name used for this sample in the LC10 scoring |
| `CTY24_confidence_nonoutlier` | Same as above but under the CROPTYPE24 schema |
| `CTY24_anomaly_flag` | Escalation category under CTY24 |
| `outlier_CTY24_cls` | Crop-type class name used for this sample in the CTY24 scoring |

> **When filtering:** only `candidate` rows are recommended for removal.  `suspect` and `flagged` should be inspected visually before dropping.

This cell works the same for both `rerun` and `update` modes — it uses whichever `LC10_flagged_gdf` / `CTY24_flagged_gdf` DataFrames are in memory.


In [ ]:
# Outer-join so samples present in only one pipeline are still included
# (e.g. a sample that has no CROPTYPE24 label still gets its LC10 score)

if LC10_flagged_gdf.empty and CTY24_flagged_gdf.empty:
    print("Both LC10 and CTY24 scored DataFrames are empty — nothing to merge.")
    merged_scores = pd.DataFrame(columns=["ref_id", "sample_id"] + ANOMALY_COLS)
elif LC10_flagged_gdf.empty:
    merged_scores = CTY24_flagged_gdf.copy()
elif CTY24_flagged_gdf.empty:
    merged_scores = LC10_flagged_gdf.copy()
else:
    merged_scores = CTY24_flagged_gdf.merge(
        LC10_flagged_gdf, on=["ref_id", "sample_id"], how="outer"
    )

print(f"Merged scores: {len(merged_scores):,} rows, {merged_scores.shape[1]} columns")
if not merged_scores.empty:
    display(merged_scores.head())


## 8b) Post-processing: skip-class fill + LC10→CTY24 escalation

Two adjustments applied to `merged_scores` **after** scoring, before writing to disk:

### Skip-class fill
Samples mapped to a skip class (e.g. `"ignore"`) are held aside by `run_pipeline` and return `NaN` in all anomaly columns.  They are definitively **not** outliers, so their scores are filled with `confidence_nonoutlier = 1.0` and `anomaly_flag = "normal"` for both domains.

### LC10 → CTY24 escalation voting
When a sample's LC10 label is `temporary_crops` **and** its LC10 anomaly flag is strictly higher than its CTY24 flag **and** CTY24 is not already `"normal"`, the CTY24 flag is raised by one level (capped at the LC10 level).  Confidence is averaged between the two domains for escalated rows.  This flows strictly from the parent class (`temporary_crops`) to its sub-crop categories, never in the reverse direction, and never promotes a `normal` CTY24 result.


In [ ]:

# ============================================================
# Post-processing helpers (shared by both Presto and AlphaEarth workflows)
# ============================================================

_FLAG_ORDER = ["normal", "flagged", "suspect", "candidate"]
_FLAG_RANK  = {f: i for i, f in enumerate(_FLAG_ORDER)}

# Terminal states that are NOT rungs on the severity ladder.  The detector could
# not form an opinion (slice too small / embedding rejected / ewoc_code absent
# from the legend) or was told to skip the row.
#
# These must never be folded into "normal".  A plain `.map(_FLAG_RANK).fillna(0)`
# would do exactly that, which is the conflation these states exist to remove:
# "we did not look" would silently read as "we looked and it is fine".  They are
# also excluded from escalation - you cannot raise a verdict that was never made.
_NON_JUDGED_FLAGS = {"unscored", "unscorable", "unmapped", "skipped"}


def _is_judged(flag_series):
    """True where the detector actually reached a verdict."""
    return ~flag_series.astype(str).isin(_NON_JUDGED_FLAGS)

# ── Post-processing skip classes ──────────────────────────────────────
# Any sample whose LC10 class OR CTY24 class appears in this list will
# have its scores reset to confidence=1.0 / flag="normal" for the
# matching domain AFTER scoring.  Independent of run_pipeline skip_classes.
# Examples: ["built-up", "water", "bare_soil", "ignore"]
POST_PROCESSING_SKIP_CLASSES: list[str] = []   # ← edit as needed


def _fill_skip_class_scores(df: pd.DataFrame) -> pd.DataFrame:
    """Fill NaN anomaly values (skip-class rows) with confidence=1.0 / flag='normal'."""
    df = df.copy()
    for prefix in ("LC10", "CTY24"):
        conf_col = f"{prefix}_confidence_nonoutlier"
        flag_col = f"{prefix}_anomaly_flag"
        if conf_col in df.columns:
            df[conf_col] = df[conf_col].fillna(1.0).astype("float32")
        if flag_col in df.columns:
            # run_pipeline now returns the explicit terminal state "skipped" for
            # skip_classes rows instead of NaN, so this fillna is a safety net
            # for older outputs only.  Deliberately NOT rewriting "skipped" (or
            # any other non-judged state) to "normal": keeping them visible is
            # the point - downstream can then choose to keep, drop or review
            # them knowingly rather than inheriting a silent "looks fine".
            df[flag_col] = df[flag_col].fillna("normal")
    return df


def _apply_postprocessing_skip_classes(
    df: pd.DataFrame,
    skip_classes: list[str],
) -> pd.DataFrame:
    """Reset scores to 1.0 / 'normal' for samples whose scored class is in skip_classes.

    Applied after scoring — for each domain independently:
    - If ``outlier_LC10_cls`` is in skip_classes → LC10 confidence=1.0, flag='normal'
    - If ``outlier_CTY24_cls`` is in skip_classes → CTY24 confidence=1.0, flag='normal'
    """
    if not skip_classes:
        return df

    df = df.copy()
    skip_set = {str(c).lower().strip() for c in skip_classes}

    domain_map = {
        "LC10":  ("outlier_LC10_cls",  "LC10_confidence_nonoutlier",  "LC10_anomaly_flag"),
        "CTY24": ("outlier_CTY24_cls", "CTY24_confidence_nonoutlier", "CTY24_anomaly_flag"),
    }

    for domain, (cls_col, conf_col, flag_col) in domain_map.items():
        if cls_col not in df.columns:
            continue
        mask = df[cls_col].astype(str).str.lower().str.strip().isin(skip_set)
        n = int(mask.sum())
        if n == 0:
            continue
        if conf_col in df.columns:
            df.loc[mask, conf_col] = float(1.0)
            df[conf_col] = df[conf_col].astype("float32")
        if flag_col in df.columns:
            df.loc[mask, flag_col] = "normal"
        print(f"[post-skip/{domain}] Reset {n:,} rows matching classes: "
              f"{df.loc[mask, cls_col].value_counts().to_dict()}")

    return df


def _apply_lc10_to_cty24_escalation(df: pd.DataFrame) -> pd.DataFrame:
    """Escalate CTY24 anomaly flag when LC10=temporary_crops is scored higher."""
    df = df.copy()

    lc10_cls_col   = "outlier_LC10_cls"
    lc10_flag_col  = "LC10_anomaly_flag"
    lc10_conf_col  = "LC10_confidence_nonoutlier"
    cty24_flag_col = "CTY24_anomaly_flag"
    cty24_conf_col = "CTY24_confidence_nonoutlier"

    required = [lc10_cls_col, lc10_flag_col, lc10_conf_col, cty24_flag_col, cty24_conf_col]
    if not all(c in df.columns for c in required):
        print("[escalation] Missing required columns — skipping LC10→CTY24 escalation.")
        return df

    lc10_rank  = df[lc10_flag_col].map(_FLAG_RANK).fillna(0).astype(int)
    cty24_rank = df[cty24_flag_col].map(_FLAG_RANK).fillna(0).astype(int)

    esc_mask = (
        (df[lc10_cls_col].astype(str).str.lower() == "temporary_crops")
        & (cty24_rank > 0)
        & (lc10_rank > cty24_rank)
    )

    n_esc = int(esc_mask.sum())
    print(f"[escalation] {n_esc:,} samples eligible for LC10→CTY24 escalation.")

    if n_esc == 0:
        return df

    new_cty24_rank = (cty24_rank + 1).clip(upper=lc10_rank)
    df.loc[esc_mask, cty24_flag_col] = new_cty24_rank[esc_mask].map(
        lambda r: _FLAG_ORDER[int(r)]
    )

    avg_conf = (
        df.loc[esc_mask, lc10_conf_col].astype(float)
        + df.loc[esc_mask, cty24_conf_col].astype(float)
    ) / 2.0
    df.loc[esc_mask, cty24_conf_col] = avg_conf.clip(0.0, 1.0).astype("float32")

    print(
        "[escalation] Escalated CTY24 flag distribution:\n"
        + df.loc[esc_mask, cty24_flag_col].value_counts().to_string()
    )
    return df


# ── Apply post-processing to merged_scores ──────────────────────────
if not merged_scores.empty:
    merged_scores = _fill_skip_class_scores(merged_scores)
    if POST_PROCESSING_SKIP_CLASSES:
        merged_scores = _apply_postprocessing_skip_classes(merged_scores, POST_PROCESSING_SKIP_CLASSES)
    merged_scores = _apply_lc10_to_cty24_escalation(merged_scores)
    print(f"\nPost-processing complete — {len(merged_scores):,} rows.")
    print("\nLC10_anomaly_flag after post-processing:")
    print(merged_scores["LC10_anomaly_flag"].value_counts(dropna=False))
    print("\nCTY24_anomaly_flag after post-processing:")
    print(merged_scores["CTY24_anomaly_flag"].value_counts(dropna=False))

    # Coverage report: how much of the collection the detector could actually
    # judge.  A large non-judged share is a finding in its own right (sparse
    # regions, encoder failures, legend gaps) and used to be invisible because
    # those rows were reported as "normal".
    for _p in ("LC10", "CTY24"):
        _fc = f"{_p}_anomaly_flag"
        if _fc in merged_scores.columns:
            _nj = merged_scores[_fc].astype(str).isin(_NON_JUDGED_FLAGS)
            print(f"\n[{_p}] not judged: {_nj.sum():,} / {len(merged_scores):,} "
                  f"({_nj.mean():.2%})")
            if _nj.any():
                print(merged_scores.loc[_nj, _fc].value_counts().to_string())
else:
    print("merged_scores is empty — post-processing skipped.")


## 8c) Add lat/lon + Export merged scores as GeoParquet (QGIS-ready)

Looks up `lat`/`lon` from the embeddings DuckDB cache, attaches them to `merged_scores`, and writes a GeoParquet file that can be opened directly in **QGIS**, shared with collaborators, or loaded with `geopandas.read_parquet()`.

Set `GEOPARQUET_CRS` to change the output CRS (default: `EPSG:4326` — WGS84 lat/lon).


In [ ]:

import geopandas as gpd
from shapely.geometry import Point

# ── Output path ────────────────────────────────────────────────────────
GEOPARQUET_SCORES_PATH = MERGED_SCORES_PATH.parent / (MERGED_SCORES_PATH.stem + "_geo.geoparquet")
GEOPARQUET_CRS = "EPSG:4326"   # WGS84 — change to reproject (e.g. "EPSG:3857")

# ── Fetch lat/lon from embeddings DuckDB ──────────────────────────────
_scores = pd.read_parquet(str(MERGED_SCORES_PATH))

if "lat" not in _scores.columns or "lon" not in _scores.columns:
    print("Looking up lat/lon from embeddings cache …")
    _con = duckdb.connect(str(EMBEDDINGS_DB_PATH), read_only=True)
    try:
        _ids = _scores[["sample_id"]].drop_duplicates()
        _con.register("_ids", _ids)
        _latlon = _con.execute(
            "SELECT e.sample_id, e.lat, e.lon "
            "FROM embeddings_cache e "
            "INNER JOIN _ids i ON e.sample_id = i.sample_id"
        ).fetchdf()
    finally:
        _con.close()
    _scores = _scores.merge(_latlon, on="sample_id", how="left")
    n_matched = _scores[["lat", "lon"]].notna().all(axis=1).sum()
    print(f"  Matched {n_matched:,} / {len(_scores):,} rows with coordinates.")

# ── Build & write GeoDataFrame ────────────────────────────────────────
if "lat" in _scores.columns and "lon" in _scores.columns:
    _df_geo = _scores.dropna(subset=["lat", "lon"]).copy()
    n_missing = len(_scores) - len(_df_geo)
    if n_missing:
        print(f"  Dropped {n_missing:,} rows with null lat/lon.")

    gdf_scores = gpd.GeoDataFrame(
        _df_geo,
        geometry=[Point(lon, lat) for lon, lat in zip(_df_geo["lon"], _df_geo["lat"])],
        crs=GEOPARQUET_CRS,
    )
    gdf_scores.to_parquet(str(GEOPARQUET_SCORES_PATH))
    print(f"GeoParquet written : {GEOPARQUET_SCORES_PATH}")
    print(f"  Rows    : {len(gdf_scores):,}")
    print(f"  CRS     : {gdf_scores.crs}")
    print(f"\n  LC10_anomaly_flag:")
    print(gdf_scores["LC10_anomaly_flag"].value_counts(dropna=False).to_string())
    print(f"\n  CTY24_anomaly_flag:")
    print(gdf_scores["CTY24_anomaly_flag"].value_counts(dropna=False).to_string())
    print(f"\n→ Open in QGIS: drag-and-drop {GEOPARQUET_SCORES_PATH.name} into the QGIS window.")
else:
    print("Skipping GeoParquet export — no lat/lon available in embeddings cache.")


In [ ]:
# Sort for deterministic ordering and write to disk for reuse in later sessions
if merged_scores.empty:
    print("No scores to write — skipping.")
else:
    merged_scores.sort_values(["ref_id", "sample_id"], inplace=True)
    merged_scores.reset_index(drop=True, inplace=True)
    merged_scores.to_parquet(str(MERGED_SCORES_PATH), index=False)
    print(f"Merged scores written to: {MERGED_SCORES_PATH}")


## 9) Write scores back to long-format parquets

For each long-format parquet file, left-join the merged anomaly scores on `(ref_id, sample_id)` and write the result.

**In `geoparquet` mode (VM):** files are updated **in-place** inside `INPUT_LONG_DIR` using an atomic temp-file swap.  The `geo` metadata block is preserved so the files remain valid GeoParquet for GIS tools.

**In `parquet` mode (HPC):** output goes to `OUTPUT_LONG_DIR` (a separate directory).

**In `update` mode:** only files containing `ref_id`s that were re-scored are rewritten (much faster).  In `rerun` mode, all files are rewritten.


In [ ]:

# Read the merged scores back from disk (if not already in memory from earlier steps)
print(f"Reading merged scores from disk...{MERGED_SCORES_PATH}")
merged_scores = pd.read_parquet(str(MERGED_SCORES_PATH))

if merged_scores.empty:
    print("No scores to write back : skipping.")
else:
    if INPUT_FORMAT == "geoparquet":
        print(f"Mode: geoparquet : in-place update in {INPUT_LONG_DIR}")
    else:
        OUTPUT_LONG_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Mode: parquet : writing to {OUTPUT_LONG_DIR}")

    # In update mode: only rewrite files for ref_ids that were rescored.
    # In rerun mode: rewrite all files (only_affected_ref_ids=None).
    only_affected = RESCORED_REF_IDS if PIPELINE_MODE == "update" else None
    if only_affected is not None:
        print(f"Update mode: limiting write-back to {len(only_affected):,} affected ref_ids")
    print(f"Only affected ref_ids: {only_affected}" if only_affected is not None else "None (all)")
    n_written = merge_scores_to_long_parquets(
        scored_df=merged_scores,
        long_parquet_dir=INPUT_LONG_DIR,
        output_parquet_dir=OUTPUT_LONG_DIR,
        anomaly_cols=ANOMALY_COLS,
        parquet_glob=PARQUET_GLOB,
        only_affected_ref_ids=only_affected,
    )
    print(f"Done : {n_written} parquet file(s) written/updated.")


## 10) Write scores back to merged wide parquet (Arrow streaming)

Joins the merged anomaly scores into the single merged wide parquet file via a streaming PyArrow pass — the full file is never held in memory at once.

**Rerun mode** (output does not exist yet): reads the *source* merged wide parquet, left-joins all scores, and writes the output from scratch.

**Update mode** (output already exists): reads the *existing output* file, patches only the rows whose `sample_id` is in the newly scored set, and rewrites atomically via a temp file.  Rows not in the new scores keep their existing anomaly values.


In [ ]:
# Read the merged scores back from disk (if not already in memory from earlier steps)
merged_scores = pd.read_parquet(str(MERGED_SCORES_PATH))

if merged_scores.empty:
    print("No scores to write back — skipping wide parquet update.")
else:
    # Build a small lookup: only the join keys + anomaly columns
    scores_lookup = (
        merged_scores[["ref_id", "sample_id"] + ANOMALY_COLS]
        .drop_duplicates(subset="sample_id")
        .copy()
    )

    # ------------------------------------------------------------------
    # Incremental update: output already exists → patch only newly scored rows
    # ------------------------------------------------------------------
    if OUTPUT_WIDE_WITH_SCORES_PATH.exists() and PIPELINE_MODE == "update":
        print(f"[wide-update] Incremental update — patching {len(scores_lookup):,} "
              f"sample scores into {OUTPUT_WIDE_WITH_SCORES_PATH}")

        scored_ids = set(scores_lookup["sample_id"].unique())
        existing_pf = pq.ParquetFile(str(OUTPUT_WIDE_WITH_SCORES_PATH))
        existing_schema = existing_pf.schema_arrow
        print(f"[wide-update] Existing output: {existing_pf.metadata.num_rows:,} rows, "
              f"{len(existing_schema)} cols")

        # Ensure schema has all anomaly columns
        extra_fields = []
        for col in ANOMALY_COLS:
            if col not in existing_schema.names:
                dtype = scores_lookup[col].dtype
                arrow_type = pa.float32() if str(dtype).startswith("float") else pa.string()
                extra_fields.append(pa.field(col, arrow_type))
        target_schema = pa.schema(list(existing_schema) + extra_fields) if extra_fields else existing_schema

        tmp_path = OUTPUT_WIDE_WITH_SCORES_PATH.with_suffix(".tmp.parquet")
        writer = pq.ParquetWriter(
            str(tmp_path), target_schema, compression="zstd",
            use_dictionary=True, write_statistics=True,
        )
        rows_written = 0
        rows_updated = 0
        try:
            for batch in existing_pf.iter_batches(batch_size=MERGE_BATCH_ROWS):
                df_batch = batch.to_pandas()
                mask = df_batch["sample_id"].isin(scored_ids)
                n_hits = int(mask.sum())
                if n_hits > 0:
                    matched = df_batch.loc[mask].copy()
                    cols_to_drop = [c for c in ANOMALY_COLS if c in matched.columns]
                    if cols_to_drop:
                        matched.drop(columns=cols_to_drop, inplace=True)
                    matched = matched.merge(scores_lookup, on=["ref_id", "sample_id"], how="left")
                    unmatched = df_batch.loc[~mask]
                    df_batch = pd.concat([unmatched, matched], ignore_index=True)
                    df_batch.sort_values("sample_id", inplace=True)
                    rows_updated += n_hits

                for col in ANOMALY_COLS:
                    if col not in df_batch.columns:
                        df_batch[col] = float("nan")

                tbl = pa.Table.from_pandas(df_batch, schema=target_schema, safe=False)
                writer.write_table(tbl, row_group_size=MERGE_ROW_GROUP_SIZE)
                rows_written += len(df_batch)
                del df_batch, tbl, batch
                gc.collect()
            print(f"[wide-update] {rows_written:,} rows written, "
                  f"{rows_updated:,} updated → {OUTPUT_WIDE_WITH_SCORES_PATH}")
        finally:
            writer.close()

        # Atomic rename
        tmp_path.replace(OUTPUT_WIDE_WITH_SCORES_PATH)

    # ------------------------------------------------------------------
    # Full write: output doesn't exist yet → read source, join all scores
    # ------------------------------------------------------------------
    else:
        print(f"Full write mode: output doesn't exist yet → reading source and joining all scores")
        print(MERGED_WIDE_PATH)
        src_pf = pq.ParquetFile(str(MERGED_WIDE_PATH))
        src_schema = src_pf.schema_arrow
        total_wide_rows = src_pf.metadata.num_rows
        print(f"Source merged wide parquet: {total_wide_rows:,} rows, {len(src_schema)} columns")

        extra_fields = []
        for col in ANOMALY_COLS:
            # Always re-add anomaly cols from scores_lookup dtype.
            # base_fields already strips them from src_schema to avoid duplicates,
            # so we must add them here unconditionally (even if they were in src_schema).
            dtype = scores_lookup[col].dtype
            arrow_type = pa.float32() if str(dtype).startswith("float") else pa.string()
            extra_fields.append(pa.field(col, arrow_type))

        base_fields = [f for f in src_schema if f.name not in ANOMALY_COLS]
        target_schema = pa.schema(base_fields + extra_fields)

        writer = pq.ParquetWriter(
            str(OUTPUT_WIDE_WITH_SCORES_PATH), target_schema, compression="zstd",
            use_dictionary=True, write_statistics=True,
        )
        rows_written = 0
        try:
            for batch in src_pf.iter_batches(batch_size=MERGE_BATCH_ROWS):
                df_batch = batch.to_pandas()
                cols_to_drop = [c for c in ANOMALY_COLS if c in df_batch.columns]
                if cols_to_drop:
                    df_batch.drop(columns=cols_to_drop, inplace=True)
                df_batch = df_batch.merge(scores_lookup, on=["ref_id", "sample_id"], how="left")
                tbl = pa.Table.from_pandas(df_batch, schema=target_schema, safe=False)
                writer.write_table(tbl, row_group_size=MERGE_ROW_GROUP_SIZE)
                rows_written += len(df_batch)
                del df_batch, tbl, batch
                gc.collect()
            print(f"Wide parquet with scores written — {rows_written:,} rows "
                  f"→ {OUTPUT_WIDE_WITH_SCORES_PATH}")
        finally:
            writer.close()


In [ ]:
import pandas as pd 
# final_scores = pd.read_parquet(OUTPUT_WIDE_WITH_SCORES_PATH)
final_scores = pd.read_parquet("/path/to/TestFolder/wc_outliers/data_for_outlier/CACHED_WIDE_MERGED/cached_wide_merged/worldcereal_all_extractions_wide_month_new_model_with_anomalies.parquet")

In [ ]:

# Compute the 'region' column for final_scores using the same spatial-join logic
# used in worldcereal.train.data.load_data_for_finetuning.
# Requires 'lat' and 'lon' columns (present in the wide-format parquet).

from worldcereal.train.data import _attach_regions_from_boundaries, _BOUNDARIES_PATH

if "region" not in final_scores.columns or final_scores["region"].isna().sum() > 0:
    print(f"Assigning regions via spatial join ({_BOUNDARIES_PATH}) ...")
    final_scores = _attach_regions_from_boundaries(
        final_scores,
        boundaries_path=_BOUNDARIES_PATH,
    )
    print("Done. Region value counts:")
else:
    print("'region' column already present — skipping spatial join.")

print(final_scores["region"].value_counts())


In [ ]:
# final_scores.to_parquet(OUTPUT_WIDE_WITH_SCORES_PATH, row_group_size=MERGE_ROW_GROUP_SIZE)
final_scores.to_parquet("/path/to/TestFolder/wc_outliers/data_for_outlier/CACHED_WIDE_MERGED/cached_wide_merged/worldcereal_all_extractions_wide_month_new_model_with_anomalies.parquet", row_group_size=MERGE_ROW_GROUP_SIZE)

In [ ]:
# split and save by region for easier loading in the next notebook into /projects/worldcereal/data/cached_wide_merged/Region_wise_files
from pathlib import Path    
MERGE_ROW_GROUP_SIZE = 100_000
outfolder = Path("/projects/worldcereal/data/cached_wide_merged/Region_wise_files_newmodel")
outfolder.mkdir(parents=True, exist_ok=True)
for region, group in final_scores.groupby("region"):
    region = region.replace(" ", "_")
    out_path = outfolder / f"{region}.parquet"
    group.to_parquet(out_path, index=False, row_group_size=MERGE_ROW_GROUP_SIZE, compression="zstd")
    print(f"Saved {len(group):,} rows for region '{region}' to: {out_path}")
    

## Optional: run the full pipeline as a script

All 10 pipeline steps (long → wide, merge, embeddings, class mappings, LC10 scoring, CTY24 scoring, score merge, write back to long parquets, write back to wide parquet) are also available as a single command-line script (`scripts/misc/compute_anomaly_scores.py`).

The cell below constructs the equivalent argument list from the parameters set in section 1 and shows the full command.  Uncomment the `run_pipeline_script(*args)` call to actually execute it — useful for running the heavy computation in a non-interactive context (e.g. a SLURM job or `screen` session).

> **Tip:** Run `python compute_anomaly_scores.py --help` to see all available flags.

## Optional: run the full pipeline as a script

All pipeline steps are also available as a single command-line script (`scripts/misc/compute_anomaly_scores.py`).

It supports two modes via `--mode`:
- **`rerun`** (default): full scoring of all points from the DuckDB embeddings cache.
- **`update`**: incremental — scans output parquets for NaN anomaly scores, computes the H3 impact zone, loads only those embeddings, and re-scores only the affected slices.

**Rerun example (HPC):**
```bash
python compute_anomaly_scores.py --mode rerun \
    --input-long-dir /projects/worldcereal/data/worldcereal_all_extractions.parquet
```

**Update example (VM, geoparquet, skip embeddings):**
```bash
python compute_anomaly_scores.py \
    --mode update \
    --input-format geoparquet \
    --input-long-dir /data/.../MERGED_PARQUETS_PHASEII_WITH_ANOMALY \
    --embeddings-db-path /data/.../embeddings_cache_LANDCOVER10_updated.duckdb \
    --wide-dir /data/.../cached_wide_parquets \
    --merged-wide-path /data/.../worldcereal_all_extractions_wide_month.parquet \
    --sp-env-file ~/.sharepointenv \
    --skip-embeddings
```

> **Tip:** Run `python compute_anomaly_scores.py --help` to see all available flags.
